### T7 · Flash Attention Forward（综合）
- **描述**：`O = softmax(Q @ K^T * scale) @ V`，在线 Softmax 版本。每个 CTA 负责一行 block（`BM × d` 的 Q），沿 N 维循环流式扫描 K/V block，中间量 `S/P` 全程驻留寄存器，不落 HBM。
- **要点**：
  1. `scale = 1/sqrt(d)` 作为 `cutlass.Float32` 预计算传入；Q 循环外 `local_tile` + `partition_A` + `make_fragment_A` 一次加载常驻（G2R）；
  2. 主循环内：K/V 经 `local_tile`（`proj=(None,1,None)`，`n_blk` 为循环变量）→ G2S 拷贝 → `partition_B` + `make_fragment_B`（S2R），`cute.gemm(tiled_mma, tCrS, tCrQ, tCrK, tCrS)` 得 `S`；
  3. Online Softmax 状态机：`m_new = max(m_prev, warp_reduce_row_max(tCrS))`；`alpha = exp(m_prev - m_new)`；`O *= alpha`；`l_new = alpha * l_prev + warp_reduce_row_sum(exp(tCrS - m_new))`——行归约必须手写 warp shuffle，掩码按你的 MMA atom 布局推导，不能直接 `.max(dim=1)`；
  4. GEMM 2 前显式降级：`tCrP.to(cutlass.Float16)`（`mma.sync` 契约要求 A/B 为 16-bit），然后 `cute.gemm(tiled_mma, tCrO, tCrP_fp16, tCrV, tCrO)` 原地累加；
  5. 循环结束后 `O /= l_prev` 归一化，`partition_C` 视角写回 gO。
- **验收**：对 `torch.nn.functional.scaled_dot_product_attention`（无 mask 版），atol=2e-2（FP16 累加误差）；`ncu` 检查 Registers/Thread 无 spill。
- **结构提示**：这是 05 篇 GEMM 分块 + 本篇行归约的叠加。先跑通 `BM=BN=64` 无流水线版本，再加 2-stage cp.async 流水线。

In [ ]:
# ============================================================================
# T7 · Flash Attention Forward（Online Softmax 版）
#   O = softmax(Q @ K^T * scale) @ V
#
# 分块策略：每个 CTA 负责一个 Q 行 block（BM × d），沿 N 维循环流式扫描
#           K/V block（BN 列），S/P 中间量全程驻留寄存器，不落 HBM。
#
# 结构：05 篇 GEMM 分块 + 本篇行归约（warp shuffle 手写）的叠加。
#       先跑通 BM=BN=64 无流水线版本，再加 2-stage cp.async 流水线。
# ============================================================================

import torch
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

# ---------------------------------------------------------------------------
# 可调参数
# ---------------------------------------------------------------------------
BM = 64          # Q 行 block 大小
BN = 64          # K/V 列 block 大小（内层循环步长）
D  = 64          # head_dim，配合 mma atom 选择（d=64 → MmaTraits 用 16x8x16）

# ============================================================================
# Kernel：grid = (cdiv(S, BM), B * H)
#   blockIdx.x → 本 CTA 负责的 Q 行 block
#   blockIdx.y → 本 CTA 负责的 (batch, head)
# ============================================================================
@cute.kernel
def flash_attn_fwd_kernel(
    gQ: cute.Tensor,            # (S, d) 切片后的 Q 视图（已按 b*h 偏移）
    gK: cute.Tensor,            # (S_kv, d)
    gV: cute.Tensor,            # (S_kv, d)
    gO: cute.Tensor,            # (S, d)
    scale: cutlass.Float32,     # 1/sqrt(d)，host 侧预计算
):
    # ---------- 0. 索引与块坐标 ----------
    # TODO: 取 tidx / bidx / bidx_y；把 gQ/gO 平移到当前 (b*h, 行块) 的起点，
    #       gK/gV 平移到当前 (b*h, 0)。
    bidx,bidy,_ = cute.arch.block_idx()
    tidx,_,_ = cute.arch.thread_idx()
    gQ = cute.local_tile(gQ,(BM,BN,D),(bidx,None,None),)
    gO = cute.local_tile(gO,(BM,BN,D),(bidx,bidy,None),)
    # gK = cute.local_tile(gK,(BM,BN,D),(None,bidy,None),)
    # gV = cute.local_tile(gV,(BM,BN,D),(None,bidy,None),)

    # ---------- 1. Shared Memory 布局声明（循环外） ----------
    # TODO: 声明 sQ (BM × d)、sK (d × BN)、sV (BN × d) 的 smem 布局，
    #       注意：sK 给 GEMM1 的 B 操作数、sV 给 GEMM2 的 B 操作数，
    #       布局要和各自 tiled_mma 的分区方式对齐（谁转置谁不转置想清楚）。
    # TODO: cute.arch.alloc_smem / 动态 smem 申请。
    
    #sQ_ptr = cute.arch.alloc_smem(cutlass.Float16,BM*D*2)?api用法?
    # 采用util方法
    smem = cutlass.utils.SmemAllocator()
    smem_layout_Q = cute.make_layout((BM,D),(BM,1)) # row-m
    smem_layout_K = cute.make_layout((D,BN),(1,BN)) # transpose
    smem_layout_V = cute.make_layout((D,BN),(BN,1)) # row major
    sQ = smem.allocate_tensor(cutlass.Float16,smem_layout_Q)
    sK = smem.allocate_tensor(cutlass.Float16,smem_layout_K)
    sV = smem.allocate_tensor(cutlass.Float16,smem_layout_V)
    

    # ---------- 2. TiledMMA 构造（循环外，一次） ----------
    # TODO: 用 mma.sync alu16x8x16 f32hf16hf16f32 构造 MMA_Atom；
    #       tile_to_shape 成 (BM, BN) 规模的 tiled_mma（GEMM1/GEMM2 可共用，
    #       但注意两者 N 维度不同时的 tile 选择）。
    mma_atom = cute.make_mma_atom(cute.nvgpu.warp.MmaF16BF16Op(
                                        cutlass.Float16,
                                        acc_dtype=cutlass.Float32,
                                        shape_mnk=(16,8,16),),
                               
    )
    # tiled_mma_1 = cute.make_tiled_mma(mma_atom,(4,8,4),prmutation_mnk=(BM,BN,D))
    # tiled_mma_2 = cute.make_tiled_mma(mma_atom,(4,8,4),prmutation_mnk=(BM,D,BN))
    # 二者一致
    tiled_mma = cute.make_tiled_mma(mma_atom,(4,8,4),prmutation_mnk=(BM,D,BN))

    # ---------- 3. Q 加载：G2R 一次常驻（要点 1） ----------
    # Q 全程不重新加载，所以直接 local_tile → partition_A → make_fragment_A，
    # 然后 copy 一次到寄存器 fragment tCrQ。
    # TODO:
    #   tQgQ = local_tile(gQ, (BM, d), ...)
    #   tCsQ / sQ 布局准备（若走 G2S→S2R 两段，则先 G2S 进 sQ 再 S2R；
    #   也可以对 Q 直接 G2R，二选一，注意与 K/V 路径一致性）
    #   thr_mma = tiled_mma.get_slice(tidx)
    #   tCrQ = thr_mma.make_fragment_A(分区后的 Q) → copy 到寄存器

    thr_mma = tiled_mma.get_slice(tidx)
    tCsQ = thr_mma.partition_A(sQ)
    tCgO = thr_mma.partition_C(gO)
    tCrQ = thr_mma.make_fragment_A(tCsQ)
    tCrO = thr_mma.make_fragment_C(tCgO)

    # TMA?
    # cp.async?
    # 同步版本
    #??这个怎么写
    copy_atom = cute.make_copy_atom(op=cute.nvgpu.CopyUniversalOp(),
                                            tensor_dtype= cutlass.Float16,
                                            num_bits_per_copy=128)
    # 1: g2s
    # 128->64fp16->64*64*2/128次就行
    tiled_copy_g2s = cute.make_tiled_copy_tv(copy_atom,thr_layout=(2,2),val_layout=(BM,D))
    thr_copy_g2s = tiled_copy_g2s.get_slice(tidx)
    tSgQ = thr_copy_g2s.partition_S(gQ)
    tDsQ = thr_copy_g2s.partition_D(sQ)
    cute.copy(thr_copy_g2s,tSgQ,tDsQ)
    cute.arch.sync_threads()
    # 2: s2r
    tiled_copy_s2r = cute.make_tiled_copy_A(copy_atom,tiled_mma)
    thr_copy_s2r = tiled_copy_s2r.get_slice(tidx)
    tSsQ = thr_copy_s2r.partition_S(sQ) #sourse
    tSrQ = thr_copy_s2r.retiled(tCrQ) # dst:mma->copy
    cute.copy(tiled_copy_s2r,tSsQ,tSrQ) 
    cute.arch.sync_threads()

    # ---------- 4. Online Softmax 状态初始化（循环外） ----------
    # TODO: m_prev = -inf  (fragment, 每线程持有自己负责的行 max)
    #       l_prev = 0.0
    #       tCrO   = zeros   (accA fragment, BM × d，累加 O)
    # 注意：m/l 是"每行"标量，但在 fragment 里是按 MMA 布局分散在各线程的，
    #       初始化形状必须和后续 warp_reduce 的输出对齐。
    m_prev = -cutlass.Float16.inf
    l_prev = cutlass.Float16(0.0)
    tCrO.fill(0.0)

    # ---------- 5. 主循环：沿 N 维扫描 K/V ----------
    for n_blk in cutlass.range(cute.cdiv(gK.shape[0], BN)):
    #
    #   ---- 5.1 K/V 流式加载：G2S ----
    #   TODO: tKgK = local_tile(gK, (d, BN), proj=(None, 1, None), (None, n_blk))
    #         tVgV = local_tile(gV, (BN, d), proj=(None, 1, None), (None, n_blk))
    #         （建议：升级版用 cp.async + 2-stage 流水线，先跑同步版）
    #         copy 到 sK / sV，sync



    #
    #   ---- 5.2 S = Q @ K^T * scale ----
    #   TODO: tCrK = thr_mma.make_fragment_B(partition_B(sK))   # S2R
    #         cute.gemm(tiled_mma, tCrS, tCrQ, tCrK, tCrS)      # 累加到 tCrS
    #         （tCrS 需先清零或作为 C 操作数初始化）
    #         TODO: tCrS 逐元素乘 scale
    #
    #   ---- 5.3 Online Softmax 状态机（要点 3，核心）----
    #   TODO:
    #     m_new = max(m_prev, warp_reduce_row_max(tCrS))
    #         # 行归约必须手写 warp shuffle：按你的 MMA atom 布局推导，
    #         # 同一行的数据分散在哪几条 lane / 哪些元素位置 →
    #         # 用 cute.arch.shuffle_sync / shfl_xor / shfl_down 分层归约，
    #         # 掩码 (lane mask / row 分组) 自己推，禁止 .max(dim=1) 偷懒
    #     alpha = exp(m_prev - m_new)
    #     tCrO *= alpha          # 用 exp 的 rescale 修正旧 O
    #     P = exp(tCrS - m_new)  # 逐元素，就地改 tCrS
    #     l_new = alpha * l_prev + warp_reduce_row_sum(P)
    #         # 同样手写 shuffle 求和，掩码与 max 版一致
    #     m_prev = m_new; l_prev = l_new
    #
    #   ---- 5.4 GEMM 2：O += P @ V（要点 4，显式降级）----
    #   TODO: tCrP_fp16 = tCrP.to(cutlass.Float16)   # mma.sync 要求 16-bit A/B
    #         tCrV = thr_mma.make_fragment_B(partition_B(sV))
    #         cute.gemm(tiled_mma, tCrO, tCrP_fp16, tCrV, tCrO)  # 原地累加
    #
    # （主循环结束）


    # ---------- 6. 收尾：归一化 + 写回（要点 5） ----------
    # TODO: tCrO /= l_prev   （l_prev 广播回 fragment 各元素，注意形状对齐）
    #       tCgO = thr_mma.partition_C(gO)
    #       copy(tCrO → tCgO)，用 cute.autovectorize 或逐元素赋值
    #       注意边界：S 不是 BM 倍数时的谓词（测试用例可先只测整除形状）


# ============================================================================
# Warp 行归约辅助函数（要点 3 的核心，必须手写 shuffle）
# ============================================================================
@cute.jit
def warp_reduce_row_max(tCrS: cute.Tensor):
    """对 fragment 中属于同一输出行的元素做 warp 内 max。
    实现提示：
      1. 先确定你的 mma atom（如 16x8x16）下 C fragment 的线程-元素映射；
      2. 同一行分散在若干 lane 上 → 用 shfl_xor 按二进制位分组归约；
      3. 归约后每个"行代表 lane"持有该行 max，其余 lane 的值无意义（或复制）；
      4. 返回形状需与 m_prev / alpha 的使用方式对齐。
    TODO: 实现归约逻辑
    """
    ...

@cute.jit
def warp_reduce_row_sum(tCrP: cute.Tensor):
    """同上，做 sum。掩码/分组方式与 max 版一致。
    TODO: 实现归约逻辑
    """
    ...


# ============================================================================
# Host 侧 JIT 入口
# ============================================================================
@cute.jit
def flash_attn_fwd(
    mQ: cute.Tensor,            # (B, H, S, d) fp16
    mK: cute.Tensor,            # (B, H, S_kv, d) fp16
    mV: cute.Tensor,            # (B, H, S_kv, d) fp16
    mO: cute.Tensor,            # (B, H, S, d) fp16
    scale: cutlass.Float32,
):
    # TODO:
    #   1. 把 (B,H,S,d) 重排视角为 batch 维 (B*H) + (S,d)，
    #      用 local_tile / make_tensor 按 (bidx_y) 切出每个 CTA 的视图，
    #      或直接在 kernel 内做指针偏移
    #   2. 计算 grid = (cdiv(S, BM), B * H)，block = (256,) 或按 atom 推导
    #   3. flash_attn_fwd_kernel(...).launch(grid=..., block=...)


# ============================================================================
# Python 封装：torch.Tensor → cute.Tensor
# ============================================================================
def flash_attention_forward(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    B, H, S, d = Q.shape
    O = torch.empty_like(Q)

    # TODO:
    #   from_dlpack(Q, assumed_align=16) 等打包四个张量；
    #   mark_layout_dynamic / 编译缓存（可按需）；
    #   scale = 1/sqrt(d) → cutlass.Float32 传入；
    #   调 flash_attn_fwd(...)，返回 O
    raise NotImplementedError


# ============================================================================
# 测试
# ============================================================================
def ref_attention(Q, K, V):
    """对齐目标：torch SDPA，无 mask 版本。"""
    return torch.nn.functional.scaled_dot_product_attention(Q, K, V)

def make_inputs(B, H, S, d, seed=0):
    gen = torch.Generator(device="cuda").manual_seed(seed)
    Q = (torch.randn(B, H, S, d, device="cuda", generator=gen) * 0.5).half()
    K = (torch.randn(B, H, S, d, device="cuda", generator=gen) * 0.5).half()
    V = (torch.randn(B, H, S, d, device="cuda", generator=gen) * 0.5).half()
    return Q, K, V

ATOL = 2e-2  # FP16 累加误差容忍度

TEST_SHAPES = [
    # (B, H, S, S_kv, d) —— 先只测整除形状：S % BM == 0 且 S_kv % BN == 0
    (1, 1, 64, 64, 64),
    (1, 2, 128, 128, 64),
    (2, 4, 256, 256, 64),
    (2, 8, 512, 512, 64),
    # 进阶：S_kv > S（decoder 形状）
    (1, 2, 128, 256, 64),
]

def test_correctness():
    print(f"{'shape':<28} {'max_abs_err':<14} {'result'}")
    print("-" * 48)
    for i, (B, H, S, Skv, d) in enumerate(TEST_SHAPES):
        Q, K, V = make_inputs(B, H, S, Skv, d, seed=i)
        O_ref = ref_attention(Q, K, V)
        O     = flash_attention_forward(Q, K, V)
        assert O.shape == O_ref.shape and O.dtype == O_ref.dtype
        assert not torch.isnan(O).any(), f"[{i}] NaN in output"

        err = (O.float() - O_ref.float()).abs().max().item()
        ok = err < ATOL
        print(f"({B},{H},{S},{Skv},{d}){'':<16} {err:<14.6f} {'✅ PASS' if ok else '❌ FAIL'}")
        assert ok, f"shape {i} failed: err={err}"
    print("-" * 48)
    print("All correctness tests passed ✅")

def test_perf():
    """粗测吞吐，ncu 验收时用：检查 Registers/Thread 无 spill。"""
    import time
    B, H, S, d = 2, 16, 2048, 64
    Q, K, V = make_inputs(B, H, S, d, seed=42)
    flash_attention_forward(Q, K, V)          # warmup / 编译
    torch.cuda.synchronize()
    N = 20
    t0 = time.perf_counter()
    for _ in range(N):
        flash_attention_forward(Q, K, V)
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / N
    flops = 4 * B * H * S * S * d  # GEMM1 (2) + GEMM2 (2)
    print(f"[perf] {B}x{H}x{S}x{d}: {dt*1e3:.3f} ms, {flops/dt/1e12:.2f} TFLOPS")

test_correctness()
# test_perf()


In [ ]:
# ============================================================================
# T7 · Flash Attention Forward（Online Softmax，mma.sync 版，目标 sm_120 / 5070 Ti）
#   O = softmax(Q @ K^T * scale) @ V
#
# 说明：
#   - 5070 Ti 是消费级 Blackwell (sm_120)，没有 Hopper/B100 的 wgmma + TMA bulk
#     async copy 路径，所以沿用 mma.sync warp-level MMA（16x8x16）+ 同步 copy
#     是正确选择。
#   - 本文件把原骨架里缺失/错误的部分补全，注释里标出【原代码 bug】和
#     【调试修复 #N】方便对照调试历史。
#
# 调试修复索引（本版本已全部落地）：
#   #1  host 侧真正按 (b,h) 切片 + (B,H,S,d)->(B*H,S,d) reshape，kernel 内 2D tile
#   #2  sV 保持行主序 (BN,D)，GEMM2 的 B=V^T 用 smem 转置视图表达，不做物理转置
#   #3  GEMM1/GEMM2 各自 tiled_mma + 各自 partition
#   #4  补齐 S fragment（tCrS）的声明（经占位 smem partition 出正确 rank）
#   #5  softmax 运行统计量 m/l 全部 fp32
#   #6  P 的 fp32->fp16 不再依赖不存在的 make_fragment API，直接寄存器写 smem
#   #7  边界：本版本仍假设 S/S_kv 整除 BM/BN（测试用例保证）
#   #8  make_tiled_copy_tv 参数语义修正：thr_layout 覆盖全部线程，
#       val_layout 是"每线程"的值布局；否则 128-bit 向量化检查必炸
#       （'cannot vectorize copy to 8 elements (static strides must be 1)'）
#   #9  atom layout 改 (4,1,1)：每个 warp 独占 16 行 x 全部 64 列，
#       行统计量的 quad 级 shfl 归约才成立（同一行绝不能跨 warp）
#   #10 mma 累加器 fragment 布局：每 lane 持 2 行，每 4 个连续元素
#       [4j,4j+1]=rowA、[4j+2,4j+3]=rowB，softmax/O/rescale/epilogue 全按此分组
#   #11 S2R 拷贝用 16-bit 原子（fragment 每线程只有 2 个连续 f16，
#       装不下 128-bit 的 8 个元素，与 #8 同源的 vectorization 约束）
# ============================================================================

import torch
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

BM = 64          # Q 行 block
BN = 64          # K/V 列 block（内层循环步长）
D  = 64          # head_dim

NUM_THREADS = 128   # = 4 warp。【调试修复 #9】必须与 tiled_mma 的 atom layout
                    # (4,1,1) 对应：4 个 warp 沿 M 排布，每 warp 独占 16 行。

NEG_INF = 1e30   # fp32 安全的 -inf 替代（避免 exp(-inf - (-inf)) = nan 的边界坑）


# ============================================================================
# Warp 行归约：16x8x16 mma atom 的 C fragment 里，一个 warp 负责 16 行 x 8 列，
# 每个 lane 持有 2 行 x 2 列 = 4 个元素（见【调试修复 #10】的分组约定）。
# 同一"行"分布在 4 个 lane 上（lane l/4 == 行 / 1，即 lane 分组的 quad），
# 用 shfl_xor 在 {1,2} 两级做蝶形归约即可覆盖这 4 个 lane。
#
# 【调试修复 #9 的前提】这个 quad 级归约成立的前提是：一个 warp 独自覆盖
# 某行的"全部 64 列"。atom layout 若含 N 方向 >1 的分量（如 (4,2,1)），
# 同一行的列会被拆到多个 warp，shfl 够不到别的 warp —— 统计量只算了半行，
# 表现为 err≈0.2 量级的"少归一化了一部分 key"的温和偏差（softmax 是归一
# 化的，所以不是 NaN 也不是全错，极具迷惑性）。
# ============================================================================
@cute.jit
def warp_reduce_row_max(val: cutlass.Float32) -> cutlass.Float32:
    # 蝶形 xor 归约：lane 分组宽度 1 -> 2（一行内的 4 个 lane 恰好
    # 用两级 xor(1)/xor(2) 覆盖）
    for offset in (1, 2):
        other = cute.arch.shuffle_sync_bfly(val, offset=offset)
        val = max(val, other)
    return val


@cute.jit
def warp_reduce_row_sum(val: cutlass.Float32) -> cutlass.Float32:
    for offset in (1, 2):
        other = cute.arch.shuffle_sync_bfly(val, offset=offset)
        val = val + other
    return val


# ============================================================================
# Kernel：grid = (cdiv(S, BM), B * H)
#   blockIdx.x → Q 行 block
#   blockIdx.y → 展平后的 (batch, head) 索引
#
# 【原代码 bug #1 修正】gQ/gK/gV/gO 在 host 侧真正按 (B*H, S, d) 展平，
#   kernel 用 bidy 索引展平后的 batch-head 维，再对 (S,d) 做 2D tile。
# ============================================================================
@cute.kernel
def flash_attn_fwd_kernel(
    mQ: cute.Tensor,            # (BH, S,    d)
    mK: cute.Tensor,            # (BH, S_kv, d)
    mV: cute.Tensor,            # (BH, S_kv, d)
    mO: cute.Tensor,            # (BH, S,    d)
    scale: cutlass.Float32,
):
    # ---------- 0. 索引与块坐标 ----------
    bidx, bidy, _ = cute.arch.block_idx()   # bidx: Q 行块; bidy: 展平 (b,h)
    tidx, _, _ = cute.arch.thread_idx()

    # 先取出当前 (b,h) 对应的 2D 切片，再做 2D tiling —— 这样维度才一致。
    # 注意：Tensor.__getitem__ 的坐标只接受"整数"或"None"
    # （None = 该维整体保留），不支持 python 的 slice(":") 语法。
    gQ_bh = mQ[bidy, None, None]            # (S, d)
    gK_bh = mK[bidy, None, None]            # (S_kv, d)
    gV_bh = mV[bidy, None, None]            # (S_kv, d)
    gO_bh = mO[bidy, None, None]            # (S, d)

    gQ = cute.local_tile(gQ_bh, (BM, D), (bidx, 0))   # 本 CTA 的 Q block
    gO = cute.local_tile(gO_bh, (BM, D), (bidx, 0))   # 本 CTA 的 O block
    # gK / gV 不在这里整块切，沿 N 维流式取 —— 放到主循环里按 n_blk 切

    # ---------- 1. Shared Memory 布局 ----------
    # sQ: (BM, D) row-major，Q 加载一次后常驻
    # sK: (BN, D) row-major。【调试修复】原 sK 写成 (D,BN):(1,D) 的转置布局，
    #     与 gK (BN,D) 形状不一致，cute.copy 直接报形状错误。行主序原样存
    #     即可 —— 这恰好是 GEMM1 的 B 操作数 (N,K)=(BN,D) 的 k-major 布局，
    #     mma.sync 对 B 的 k 连续布局原生友好。
    # sV: (BN, D) row-major。【原代码 bug #2 修正】V 天然是 (BN,d)，不做物理
    #     转置；GEMM2 需要的 B=V^T (D,BN) 用"转置视图"表达（见 6.4），
    #     S2R 用 16-bit 单元素原子，任何 stride 都合法，正确性优先。
    # sP: (BM, BN) fp16，P 的 fp32->fp16 中转【原代码 bug #6 修正】
    # sS: (BM, BN) fp32，纯脚手架：只为 partition_C 出正确 rank 的 tCrS
    #     【原代码 bug #4 修正】——传 (BM,BN) tuple 给 make_fragment_C 得到
    #     的是扁平形状，和 partition 出来的 3 维 mma fragment rank 对不上。
    smem = cutlass.utils.SmemAllocator()
    sQ = smem.allocate_tensor(cutlass.Float16, cute.make_layout((BM, D), stride=(D, 1)))
    sK = smem.allocate_tensor(cutlass.Float16, cute.make_layout((BN, D), stride=(D, 1)))
    sV = smem.allocate_tensor(cutlass.Float16, cute.make_layout((BN, D), stride=(D, 1)))
    sP = smem.allocate_tensor(cutlass.Float16, cute.make_layout((BM, BN), stride=(BN, 1)))
    sS = smem.allocate_tensor(cutlass.Float32, cute.make_layout((BM, BN), stride=(BN, 1)))

    # ---------- 2. TiledMMA ----------
    # 基础 atom：mma.sync.m16n8k16.f32.f16.f16.f32（Ampere 起的标准 warp MMA）
    mma_atom = cute.make_mma_atom(
        cute.nvgpu.warp.MmaF16BF16Op(
            cutlass.Float16,
            acc_dtype=cutlass.Float32,
            shape_mnk=(16, 8, 16),
        )
    )
    # 【调试修复 #9】atom layout (4,1,1)：
    #   逻辑 tile = (4*16, 1*8, 16) = (64, 8, 16)
    #   → M 方向 4 个 warp 覆盖 64 行（每 warp 16 行）
    #   → N 方向单个 warp，靠 rest_n = 64/8 = 8 次 atom 迭代覆盖全部 64 列
    #   → 每个 warp 独占 16 行 x 全部 64 列，行统计量不跨 warp，
    #     quad 级 shfl 归约（warp_reduce_row_*）才成立。
    # 每线程 accumulator = 8列 x 2行 = 16 元素/atom x 8 次 n 迭代 = 128？不：
    # 实际每线程持有 (MMA=4, MMA_M=1, MMA_N=8) = 32 个 fp32 累加值，
    # 4 warp x 32 = 128 lane-值 组，128 线程 x 32 = 4096 = 64x64，无冗余全覆盖。
    # 【原代码 bug #3 修正】GEMM1 的 K 维是 head_dim D，GEMM2 的 K 维是 BN，
    #   两者 (M,N,K) 语义不同；虽然此处两者 tile 恰好同为 (64,64)，仍各自
    #   make_tiled_mma + 各自 partition，语义清晰且避免共享 slice 状态。
    tiled_mma_1 = cute.make_tiled_mma(mma_atom, (4, 1, 1))   # S = Q@K^T, K维=D
    tiled_mma_2 = cute.make_tiled_mma(mma_atom, (4, 1, 1))   # O += P@V,  K维=BN

    thr_mma_1 = tiled_mma_1.get_slice(tidx)
    thr_mma_2 = tiled_mma_2.get_slice(tidx)

    # ---------- 3. 拷贝原子 ----------
    # 【调试修复 #8】make_tiled_copy_tv(atom, thr_layout, val_layout) 的参数
    #   语义：thr_layout = 参与搬数的线程布局（必须覆盖整个 CTA 的线程数）；
    #         val_layout = 每个线程搬的值布局（不是整个 tile！）。
    #   向量化不变量：每线程最内维 value 数 = num_bits / dtype_bits，
    #   且在 src/dst 上都落在 stride=1 的维度。
    #   G2S：128 bit / 16 bit = 每线程内维 8 个连续 f16。
    #   128 线程 = (32,4)，每线程 (1,8) → tile (32*1, 4*8) = (32,32)，
    #   64x64 tile 由 rest 迭代补满。
    # 【调试修复 #11】S2R：A/B fragment 每线程只有 2 个连续 f16（k 方向），
    #   装不下 8 个 —— 128-bit atom 必炸同一个 vectorization 错误。
    #   用 16-bit 原子（每条指令 1 元素，任何布局合法）；k-major 操作数
    #   以后可优化到 lmatrix / 32-bit，正确性版本先保守。
    copy_atom_g2s = cute.make_copy_atom(
        op=cute.nvgpu.CopyUniversalOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )
    copy_atom_s2r = cute.make_copy_atom(
        op=cute.nvgpu.CopyUniversalOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=16,
    )

    tiled_copy_g2s_q = cute.make_tiled_copy_tv(
        copy_atom_g2s, cute.make_layout((32, 4)), cute.make_layout((1, 8)))
    tiled_copy_g2s_kv = cute.make_tiled_copy_tv(
        copy_atom_g2s, cute.make_layout((32, 4)), cute.make_layout((1, 8)))

    # ---------- 4. Q 加载：G2S -> S2R，全程常驻 ----------
    thr_copy_q = tiled_copy_g2s_q.get_slice(tidx)
    cute.copy(tiled_copy_g2s_q, thr_copy_q.partition_S(gQ), thr_copy_q.partition_D(sQ))
    cute.arch.sync_threads()

    # S2R：按 mma 的 A 操作数视角 partition sQ，读进寄存器 fragment tCrQ
    tCrQ = thr_mma_1.make_fragment_A(thr_mma_1.partition_A(sQ))
    tiled_copy_s2r_q = cute.make_tiled_copy_A(copy_atom_s2r, tiled_mma_1)
    thr_copy_s2r_q = tiled_copy_s2r_q.get_slice(tidx)
    cute.copy(tiled_copy_s2r_q,
              thr_copy_s2r_q.partition_S(sQ),
              thr_copy_s2r_q.retile(tCrQ))
    cute.arch.sync_threads()

    # ---------- 5. GEMM1 的 S fragment / GEMM2 的 O accumulator ----------
    # 【原代码 bug #4 修正】经占位 smem sS partition_C 再 make_fragment_C，
    #   拿到正确的 (atom_shape, rest_m, rest_n) 三维 fragment 结构。
    tCrS = thr_mma_1.make_fragment_C(thr_mma_1.partition_C(sS))

    # tCrO：GEMM2 的 fp32 累加器，直接对全局 gO partition（只用作结构，
    #   数值全程留寄存器，epilogue 时逐元素 cast 写回）
    tCrO = thr_mma_2.make_fragment_C(thr_mma_2.partition_C(gO))
    tCrO.fill(0.0)

    # ---------- 6. Online Softmax 状态 ----------
    # 【原代码 bug #5 修正】m/l 必须是 fp32。
    # 【调试修复 #10】mma 累加器的 canonical 布局：每个 lane 持有 2 行，
    #   每 4 个连续 fragment 元素为一组：[4j,4j+1] 属于 rowA，[4j+2,4j+3]
    #   属于 rowB（rowB = rowA + 8）。因此每线程维护两套 running stats。
    m_prev_a = cutlass.Float32(-NEG_INF)
    m_prev_b = cutlass.Float32(-NEG_INF)
    l_prev_a = cutlass.Float32(0.0)
    l_prev_b = cutlass.Float32(0.0)

    n_tiles = (gK_bh.shape[0] + BN - 1) // BN   # 手写 cdiv，不依赖版本差异

    # ---------- 7. 主循环：沿 N 维扫描 K/V ----------
    for n_blk in cutlass.range(n_tiles):
        gK = cute.local_tile(gK_bh, (BN, D), (n_blk, 0))
        gV = cute.local_tile(gV_bh, (BN, D), (n_blk, 0))

        # 7.1 K/V 流式加载 G2S（同步版；生产版改进方向：cp.async 2-stage）
        thr_copy_kv = tiled_copy_g2s_kv.get_slice(tidx)
        cute.copy(tiled_copy_g2s_kv,
                  thr_copy_kv.partition_S(gK), thr_copy_kv.partition_D(sK))
        cute.copy(tiled_copy_g2s_kv,
                  thr_copy_kv.partition_S(gV), thr_copy_kv.partition_D(sV))
        cute.arch.sync_threads()

        # 7.2 GEMM1: S = Q @ K^T * scale
        #     sK 是 (BN,D) 行主序 = B 操作数 (N,K) 的 k-major，直接 partition
        tCrK = thr_mma_1.make_fragment_B(thr_mma_1.partition_B(sK))
        cpy_s2r_k = cute.make_tiled_copy_B(copy_atom_s2r, tiled_mma_1)
        thr_s2r_k = cpy_s2r_k.get_slice(tidx)
        cute.copy(cpy_s2r_k, thr_s2r_k.partition_S(sK), thr_s2r_k.retile(tCrK))

        tCrS.fill(0.0)
        cute.gemm(tiled_mma_1, tCrS, tCrQ, tCrK, tCrS)
        for i in range(cute.size(tCrS)):
            tCrS[i] = tCrS[i] * scale

        # 7.3 Online softmax 状态机（按 rowA/rowB 两套独立统计）
        # 7.3.1 行内局部 max：每 4 元素一组的 [4j,4j+1] 归入 rowA，
        #       [4j+2,4j+3] 归入 rowB【调试修复 #10 —— 原版把两行 max 到
        #       一个标量上，统计量全错，这是 err≈0.2 的直接来源之一】
        row_max_a = cutlass.Float32(-NEG_INF)
        row_max_b = cutlass.Float32(-NEG_INF)
        for j in range(cute.size(tCrS) // 4):
            row_max_a = max(row_max_a,
                            cutlass.Float32(tCrS[4 * j + 0]),
                            cutlass.Float32(tCrS[4 * j + 1]))
            row_max_b = max(row_max_b,
                            cutlass.Float32(tCrS[4 * j + 2]),
                            cutlass.Float32(tCrS[4 * j + 3]))
        # quad 内蝶形归约：同一行的 4 个 lane 恰好持同一行的不同列
        # 【调试修复 #9 成立的前提：每 warp 独占全部 64 列】
        m_new_a = max(m_prev_a, warp_reduce_row_max(row_max_a))
        m_new_b = max(m_prev_b, warp_reduce_row_max(row_max_b))
        # cute.math.max(m_prev_a - m_new_a)
        alpha_a = cute.arch.exp(m_prev_a - m_new_a)
        alpha_b = cute.arch.exp(m_prev_b - m_new_b)

        # 7.3.2 用 alpha 修正旧的 O 累加值（online softmax 的 rescale 步骤，
        #       同样必须按行分组 —— 原版整块乘同一个 alpha 是错的）
        for j in range(cute.size(tCrO) // 4):
            tCrO[4 * j + 0] = tCrO[4 * j + 0] * alpha_a
            tCrO[4 * j + 1] = tCrO[4 * j + 1] * alpha_a
            tCrO[4 * j + 2] = tCrO[4 * j + 2] * alpha_b
            tCrO[4 * j + 3] = tCrO[4 * j + 3] * alpha_b

        # 7.3.3 exp(S - m_new) 得到未归一化权重 P，就地改写 tCrS；
        #       同步累加行和（rowA/rowB 分开）
        row_sum_a = cutlass.Float32(0.0)
        row_sum_b = cutlass.Float32(0.0)
        for j in range(cute.size(tCrS) // 4):
            pa0 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 0]) - m_new_a)
            pa1 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 1]) - m_new_a)
            pb0 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 2]) - m_new_b)
            pb1 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 3]) - m_new_b)
            tCrS[4 * j + 0] = pa0
            tCrS[4 * j + 1] = pa1
            tCrS[4 * j + 2] = pb0
            tCrS[4 * j + 3] = pb1
            row_sum_a = row_sum_a + pa0 + pa1
            row_sum_b = row_sum_b + pb0 + pb1
        l_prev_a = alpha_a * l_prev_a + warp_reduce_row_sum(row_sum_a)
        l_prev_b = alpha_b * l_prev_b + warp_reduce_row_sum(row_sum_b)
        m_prev_a = m_new_a
        m_prev_b = m_new_b

        # 7.4 P 降级到 fp16 并写入 smem
        # 【原代码 bug #6 修正】不依赖不存在的 make_fragment API；
        #   也不再走 fp32 smem 中转 —— tCrS 与 partition_C(sP) 结构同构
        #   （都是 tiled_mma_1 视角的 (64,64) C 布局），寄存器坐标与 smem
        #   坐标一一对应，直接逐元素 downcast 写入 sP。
        tCsP_w = thr_mma_1.partition_C(sP)
        for i in range(cute.size(tCrS)):
            tCsP_w[i] = cutlass.Float16(tCrS[i])
        cute.arch.sync_threads()   # 写 sP 完成，读之前必须可见

        # 7.5 GEMM2: O += P @ V
        #     A = P：从 sP 按 mma A 视角读回寄存器（16-bit S2R）
        tCrP = thr_mma_2.make_fragment_A(thr_mma_2.partition_A(sP))
        cpy_s2r_p = cute.make_tiled_copy_A(copy_atom_s2r, tiled_mma_2)
        thr_s2r_p = cpy_s2r_p.get_slice(tidx)
        cute.copy(cpy_s2r_p, thr_s2r_p.partition_S(sP), thr_s2r_p.retile(tCrP))

        #     B = V^T (D,BN)：【原代码 bug #2 修正】sV 物理上是 (BN,D)
        #     行主序，这里用 iterator + 转置 layout 造一个零拷贝的转置视图，
        #     S2R 走 16-bit 原子（任何 stride 合法，正确性优先）。
        #     生产版优化方向：物理转置 + ldmatrix.trans，或 cp.async 时转置写。
        sVT = cute.make_tensor(sV.iterator, cute.make_layout((D, BN), stride=(1, D)))
        tCrV = thr_mma_2.make_fragment_B(thr_mma_2.partition_B(sVT))
        cpy_s2r_v = cute.make_tiled_copy_B(copy_atom_s2r, tiled_mma_2)
        thr_s2r_v = cpy_s2r_v.get_slice(tidx)
        cute.copy(cpy_s2r_v, thr_s2r_v.partition_S(sVT), thr_s2r_v.retile(tCrV))

        cute.gemm(tiled_mma_2, tCrO, tCrP, tCrV, tCrO)
        cute.arch.sync_threads()   # 下一轮覆盖 sK/sV/sP 前必须同步

    # ---------- 8. 收尾：归一化 + 写回 ----------
    # 【原代码 bug #7】边界情形（S/S_kv 不整除）仍假设整除，测试用例保证；
    #   生产版必须加 predicate 逐元素 mask。
    inv_l_a = cutlass.Float32(1.0) / l_prev_a
    inv_l_b = cutlass.Float32(1.0) / l_prev_b
    # tCrO 是 fp32 accumulator，gO 是 fp16：cute.copy 不做隐式类型转换，
    # 逐元素 cast 写回【原代码 bug #7 追加修正】
    for j in range(cute.size(tCrO) // 4):
        tCgO = thr_mma_2.partition_C(gO)   # 每次引用取最新视图（编译期常量，无开销）
        tCgO[4 * j + 0] = cutlass.Float16(tCrO[4 * j + 0] * inv_l_a)
        tCgO[4 * j + 1] = cutlass.Float16(tCrO[4 * j + 1] * inv_l_a)
        tCgO[4 * j + 2] = cutlass.Float16(tCrO[4 * j + 2] * inv_l_b)
        tCgO[4 * j + 3] = cutlass.Float16(tCrO[4 * j + 3] * inv_l_b)


# ============================================================================
# Host 侧 JIT 入口
# ============================================================================
@cute.jit
def flash_attn_fwd(
    mQ: cute.Tensor,     # (BH, S,    d)
    mK: cute.Tensor,     # (BH, S_kv, d)
    mV: cute.Tensor,     # (BH, S_kv, d)
    mO: cute.Tensor,     # (BH, S,    d)
    scale: cutlass.Float32,
):
    BH, S, d = mQ.shape
    # 手写向上取整（cute.cdiv 在部分版本不存在，避免 API 演进差异）
    grid = ((S + BM - 1) // BM, BH, 1)
    # 【调试修复 #9】必须与 kernel 内 atom layout (4,1,1) 对应：4 warp
    block = (NUM_THREADS, 1, 1)
    flash_attn_fwd_kernel(mQ, mK, mV, mO, scale).launch(grid=grid, block=block)


# ============================================================================
# Python 封装：torch.Tensor -> cute.Tensor
# ============================================================================
_compiled_cache = {}

def flash_attention_forward(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    B, H, S, d = Q.shape
    Skv = K.shape[2]
    assert d == D, f"当前 kernel 硬编码 head_dim={D}，输入是 {d}"
    assert S % BM == 0 and Skv % BN == 0, \
        "当前版本假设 S % BM == 0 且 S_kv % BN == 0（边界 predicate 未实现）"

    O = torch.empty_like(Q)

    # 【原代码 bug #1 的落地处】(B,H,S,d) -> (B*H,S,d) 展平，
    # kernel 用 bidy 索引展平后的 batch-head 维。
    Qf = Q.reshape(B * H, S, d).contiguous()
    Kf = K.reshape(B * H, Skv, d).contiguous()
    Vf = V.reshape(B * H, Skv, d).contiguous()
    Of = O.reshape(B * H, S, d)

    mQ = from_dlpack(Qf, assumed_align=16)
    mK = from_dlpack(Kf, assumed_align=16)
    mV = from_dlpack(Vf, assumed_align=16)
    mO = from_dlpack(Of, assumed_align=16)

    scale = cutlass.Float32(1.0 / (d ** 0.5))

    key = (B * H, S, Skv, d)
    if key not in _compiled_cache:
        _compiled_cache[key] = cute.compile(flash_attn_fwd, mQ, mK, mV, mO, scale)
    _compiled_cache[key](mQ, mK, mV, mO, scale)

    return Of.reshape(B, H, S, d)


# ============================================================================
# 测试
# ============================================================================
def ref_attention(Q, K, V):
    return torch.nn.functional.scaled_dot_product_attention(Q, K, V)


def make_inputs(B, H, S, Skv, d, seed=0):
    gen = torch.Generator(device="cuda").manual_seed(seed)
    Q = (torch.randn(B, H, S, d, device="cuda", generator=gen) * 0.5).half()
    K = (torch.randn(B, H, Skv, d, device="cuda", generator=gen) * 0.5).half()
    V = (torch.randn(B, H, Skv, d, device="cuda", generator=gen) * 0.5).half()
    return Q, K, V


ATOL = 2e-2

TEST_SHAPES = [
    (1, 1, 64, 64, 64),
    (1, 2, 128, 128, 64),
    (2, 4, 256, 256, 64),
    (2, 8, 512, 512, 64),
    (1, 2, 128, 256, 64),
]


def test_correctness():
    print(f"{'shape':<28} {'max_abs_err':<14} {'result'}")
    print("-" * 48)
    for i, (B, H, S, Skv, d) in enumerate(TEST_SHAPES):
        Q, K, V = make_inputs(B, H, S, Skv, d, seed=i)
        O_ref = ref_attention(Q, K, V)
        O = flash_attention_forward(Q, K, V)
        assert O.shape == O_ref.shape and O.dtype == O_ref.dtype
        assert not torch.isnan(O).any(), f"[{i}] NaN in output"
        err = (O.float() - O_ref.float()).abs().max().item()
        ok = err < ATOL
        print(f"({B},{H},{S},{Skv},{d}){'':<16} {err:<14.6f} {'✅ PASS' if ok else '❌ FAIL'}")
        assert ok, f"shape {i} failed: err={err}"
    print("-" * 48)
    print("All correctness tests passed ✅")


def test_perf():
    import time
    B, H, S, d = 2, 16, 2048, 64
    Q, K, V = make_inputs(B, H, S, S, d, seed=42)
    flash_attention_forward(Q, K, V)
    torch.cuda.synchronize()
    N = 20
    t0 = time.perf_counter()
    for _ in range(N):
        flash_attention_forward(Q, K, V)
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / N
    flops = 4 * B * H * S * S * d
    print(f"[perf] {B}x{H}x{S}x{d}: {dt*1e3:.3f} ms, {flops/dt/1e12:.2f} TFLOPS")


test_correctness()
# test_perf()


shape                        max_abs_err    result
------------------------------------------------
(1,1,64,64,64)                 0.000000       ✅ PASS
(1,2,128,128,64)                 0.000122       ✅ PASS
(2,4,256,256,64)                 0.000061       ✅ PASS
(2,8,512,512,64)                 0.000061       ✅ PASS
(1,2,128,256,64)                 0.000061       ✅ PASS
------------------------------------------------
All correctness tests passed ✅


In [8]:
# ============================================================================
# T7-async · Flash Attention Forward（Online Softmax + cp.async 2-stage 双缓冲
#             流水线版，目标 sm_120 / 5070 Ti）
#   O = softmax(Q @ K^T * scale) @ V
#
# 相对同步版 T7 的改动（这正是 T7 里预留的那句注释指向的方向：
#   "K/V 流式加载 G2S（同步版；生产版改进方向：cp.async 2-stage）"）：
#
#   【流水线优化 #1】sK / sV 从单缓冲 (BN,D) 升级为双缓冲 (NSTAGES,BN,D)，
#       用一个额外的 stage 维做 ping-pong，stage 下标用 n_blk % 2 动态索引
#       （和文件里已经在用的 mQ[bidy, None, None] 动态整数索引是同一手法，
#       不是新概念）。
#   【流水线优化 #2】K/V 的 G2S 拷贝原子从 CopyUniversalOp（同步 ld/st）换成
#       cute.nvgpu.cpasync.CopyG2SOp（cp.async，发起后立即返回，不占用
#       寄存器，靠专门的异步拷贝引擎搬数）。S2R（sK/sV -> mma fragment）
#       仍然是同步的 16-bit CopyUniversalOp，不变。
#   【流水线优化 #3】主循环重排成"软件流水线"：
#       prologue 先为 tile 0 发起 cp.async 并 commit_group；
#       进入循环后，先为 tile n_blk+1 发起下一次 cp.async 并 commit_group
#       （这一步和当前 tile 的计算是可以被硬件重叠掉的 —— 发起后立刻返回，
#       不等待），然后 cp_async_wait_group(1) 只等"这次新发的之前"那一组
#       （也就是 tile n_blk 自己那组）落地，再用 sK/sV[cur] 做计算。
#       最后一块没有下一块可发，直接 cp_async_wait_group(0) 排空。
#   【流水线优化 #4】原 T7 循环尾部那句
#       "cute.arch.sync_threads()  # 下一轮覆盖 sK/sV/sP 前必须同步"
#       原本只是为了保护单缓冲 sP 的 WAR 冒险。双缓冲后它还多担了一层责任：
#       tile n_blk+2 的 prefetch 会写回 stage (n_blk)%2（即本轮 cur 用过的
#       那块 buffer），这次写发生在"下一轮"循环体最开头。只要本轮结尾这个
#       sync_threads() 还在，就能保证本轮所有线程对 sK/sV[cur] 的 S2R 读
#       （ld.shared）已经全部完成，下一轮再发起的 cp.async 写才不会跟这些
#       读踩踏。——即双缓冲不需要额外加同步点，原有那个 barrier 顺手就把
#       新增的 WAR 冒险也保护了，但这个因果关系容易被忽略，写在这里备查。
#   【流水线优化 #5】Q 依旧同步加载一次（G2S 用回 CopyUniversalOp），
#       不做流水线 —— Q 只加载一次不在热路径循环里，双缓冲收益为零，
#       反而多占一份 smem，不值得。
#
# 【实测踩过的坑】
#   - cp.async 的 128-bit 拷贝原子要求目的 memref 带 align<16>（16 字节）
#     类型属性。最早以为是"下标是运行时值，编译期证不出对齐"，改成
#     if nxt==0/else 分支里用字面量 0/1 选 memref；又以为是"scf.if 合并
#     两个分支的值时把 align 属性弄丢了"，把 cute.copy 挪进分支内部。
#     两次改完报错文本都一字不差——这就说明两个猜测都是错的，跟怎么
#     下标、跨不跨分支毫无关系。真正原因：sK_buf/sV_buf 用
#     smem.allocate_tensor(...) 分配时没有显式要求对齐，默认对齐不保证
#     凑够 16 字节；T7 里的单缓冲 sK/sV 之所以没事，纯粹是巧合对上了。
#     修法是在分配处加 byte_alignment=16（单位字节，正式签名是
#     allocate_tensor(element_type, layout, byte_alignment=1, swizzle=None)，
#     不叫 alignment——之前手滑写错过一次，报 TypeError: got an unexpected
#     keyword argument 'alignment'，改成 byte_alignment 才对），下标方式
#     恢复成最简单的 sK_buf[nxt, None, None] 就行，不需要任何 if/else。—— 教训：这类 "IR verification failed" 的对齐报错先去查
#     分配处的 alignment 参数，比在下标/控制流上绕圈子命中率高得多。
#
# 【还没实测过，仍是潜在风险点】：
#   - smem 预算：sK/sV 各自翻倍到 2 份 (BN,D) fp16 = 2*64*64*2B = 16KB，
#     两个加起来 32KB，比 T7 同步版多 16KB；64x64 fp32 的 sS 占位 buffer
#     和 sQ/sP 不变。5070 Ti 单 block 可用 smem 远够，但如果之后想把
#     BM/BN 调大，记得重新算一遍。
# ============================================================================

import torch
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

BM = 64          # Q 行 block
BN = 64          # K/V 列 block（内层循环步长）
D  = 64          # head_dim
NSTAGES = 2      # 【流水线优化 #1】cp.async 双缓冲的 stage 数

NUM_THREADS = 128   # = 4 warp，跟 tiled_mma 的 atom layout (4,1,1) 对应

NEG_INF = 1e30


@cute.jit
def warp_reduce_row_max(val: cutlass.Float32) -> cutlass.Float32:
    for offset in (1, 2):
        other = cute.arch.shuffle_sync_bfly(val, offset=offset)
        val = max(val, other)
    return val


@cute.jit
def warp_reduce_row_sum(val: cutlass.Float32) -> cutlass.Float32:
    for offset in (1, 2):
        other = cute.arch.shuffle_sync_bfly(val, offset=offset)
        val = val + other
    return val


@cute.kernel
def flash_attn_fwd_kernel(
    mQ: cute.Tensor,            # (BH, S,    d)
    mK: cute.Tensor,            # (BH, S_kv, d)
    mV: cute.Tensor,            # (BH, S_kv, d)
    mO: cute.Tensor,            # (BH, S,    d)
    scale: cutlass.Float32,
):
    # ---------- 0. 索引与块坐标 ----------
    bidx, bidy, _ = cute.arch.block_idx()
    tidx, _, _ = cute.arch.thread_idx()

    gQ_bh = mQ[bidy, None, None]
    gK_bh = mK[bidy, None, None]
    gV_bh = mV[bidy, None, None]
    gO_bh = mO[bidy, None, None]

    gQ = cute.local_tile(gQ_bh, (BM, D), (bidx, 0))
    gO = cute.local_tile(gO_bh, (BM, D), (bidx, 0))

    # ---------- 1. Shared Memory 布局 ----------
    # sQ / sP / sS 与 T7 完全一致（单缓冲，理由见文件头 #5 和 T7 原注释）。
    # sK / sV 【流水线优化 #1】多一维 stage，物理上就是把 NSTAGES 份
    # (BN,D) 行主序 buffer 首尾拼在一起，stage 维 stride = BN*D。
    smem = cutlass.utils.SmemAllocator()
    sQ = smem.allocate_tensor(cutlass.Float16, cute.make_layout((BM, D), stride=(D, 1)))
    sK_buf = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((NSTAGES, BN, D), stride=(BN * D, D, 1)),
        byte_alignment=16,   # 【调试修复 #D，见文件头】字节数，= 128 bit
    )
    sV_buf = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((NSTAGES, BN, D), stride=(BN * D, D, 1)),
        byte_alignment=16,
    )
    sP = smem.allocate_tensor(cutlass.Float16, cute.make_layout((BM, BN), stride=(BN, 1)))
    sS = smem.allocate_tensor(cutlass.Float32, cute.make_layout((BM, BN), stride=(BN, 1)))

    # ---------- 2. TiledMMA（与 T7 相同） ----------
    mma_atom = cute.make_mma_atom(
        cute.nvgpu.warp.MmaF16BF16Op(
            cutlass.Float16,
            acc_dtype=cutlass.Float32,
            shape_mnk=(16, 8, 16),
        )
    )
    tiled_mma_1 = cute.make_tiled_mma(mma_atom, (4, 1, 1))   # S = Q@K^T
    tiled_mma_2 = cute.make_tiled_mma(mma_atom, (4, 1, 1))   # O += P@V

    thr_mma_1 = tiled_mma_1.get_slice(tidx)
    thr_mma_2 = tiled_mma_2.get_slice(tidx)

    # ---------- 3. 拷贝原子 ----------
    # Q 依旧用同步 128-bit CopyUniversalOp（G2S 只跑一次，见 #5）。
    copy_atom_g2s_sync = cute.make_copy_atom(
        op=cute.nvgpu.CopyUniversalOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )
    # 【流水线优化 #2】K/V 的 G2S 换成 cp.async：发起后立即返回，配合
    # commit_group / wait_group 做流水线。thr/val layout 跟同步版保持一致
    # （128-bit 向量化的约束没有变——8 个连续 f16 一次搬）。
    # TODO：如果这里的关键字参数名和 CopyUniversalOp 对不上，去
    # `cutlass.cute.nvgpu.cpasync.CopyG2SOp.__init__` 的签名核对一下。
    copy_atom_g2s_async = cute.make_copy_atom(
        op=cute.nvgpu.cpasync.CopyG2SOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )
    copy_atom_s2r = cute.make_copy_atom(
        op=cute.nvgpu.CopyUniversalOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=16,
    )

    tiled_copy_g2s_q = cute.make_tiled_copy_tv(
        copy_atom_g2s_sync, cute.make_layout((32, 4)), cute.make_layout((1, 8)))
    tiled_copy_g2s_kv = cute.make_tiled_copy_tv(
        copy_atom_g2s_async, cute.make_layout((32, 4)), cute.make_layout((1, 8)))

    # ---------- 4. Q 加载：同步 G2S -> S2R，全程常驻 ----------
    thr_copy_q = tiled_copy_g2s_q.get_slice(tidx)
    cute.copy(tiled_copy_g2s_q, thr_copy_q.partition_S(gQ), thr_copy_q.partition_D(sQ))
    cute.arch.sync_threads()

    tCrQ = thr_mma_1.make_fragment_A(thr_mma_1.partition_A(sQ))
    tiled_copy_s2r_q = cute.make_tiled_copy_A(copy_atom_s2r, tiled_mma_1)
    thr_copy_s2r_q = tiled_copy_s2r_q.get_slice(tidx)
    cute.copy(tiled_copy_s2r_q,
              thr_copy_s2r_q.partition_S(sQ),
              thr_copy_s2r_q.retile(tCrQ))
    cute.arch.sync_threads()

    # ---------- 5. S fragment / O accumulator（与 T7 相同） ----------
    tCrS = thr_mma_1.make_fragment_C(thr_mma_1.partition_C(sS))
    tCrO = thr_mma_2.make_fragment_C(thr_mma_2.partition_C(gO))
    tCrO.fill(0.0)

    # ---------- 6. Online Softmax 状态（与 T7 相同） ----------
    m_prev_a = cutlass.Float32(-NEG_INF)
    m_prev_b = cutlass.Float32(-NEG_INF)
    l_prev_a = cutlass.Float32(0.0)
    l_prev_b = cutlass.Float32(0.0)

    n_tiles = (gK_bh.shape[0] + BN - 1) // BN
    thr_copy_kv = tiled_copy_g2s_kv.get_slice(tidx)

    # ---------- 6.5 Prologue：为 tile 0 发起第一次 cp.async 预取 ----------
    # 【流水线优化 #3】进入主循环前先把 stage 0 填上，对应经典双缓冲里
    # "prologue 预取 depth-1 块"，这里 NSTAGES=2 所以只需预取 1 块。
    gK0 = cute.local_tile(gK_bh, (BN, D), (0, 0))
    gV0 = cute.local_tile(gV_bh, (BN, D), (0, 0))
    sK0 = sK_buf[0, None, None]
    sV0 = sV_buf[0, None, None]
    cute.copy(tiled_copy_g2s_kv, thr_copy_kv.partition_S(gK0), thr_copy_kv.partition_D(sK0))
    cute.copy(tiled_copy_g2s_kv, thr_copy_kv.partition_S(gV0), thr_copy_kv.partition_D(sV0))
    cute.arch.cp_async_commit_group()

    # ---------- 7. 主循环：沿 N 维扫描 K/V（软件流水线版） ----------
    for n_blk in cutlass.range(n_tiles):
        cur = n_blk % 2
        has_next = n_blk + 1 < n_tiles

        # 7.1 【流水线优化 #3】先把下一块的 cp.async 发出去，让它跟本轮的
        #     计算（7.2~7.5）在硬件上重叠；发起后不等待，立刻往下走。
        if has_next:
            nxt = (n_blk + 1) % 2
            gK_next = cute.local_tile(gK_bh, (BN, D), (n_blk + 1, 0))
            gV_next = cute.local_tile(gV_bh, (BN, D), (n_blk + 1, 0))
            # 【调试修复 #D，根因见文件头】前两版分别怀疑"offset 不是编译期
            # 常量"和"scf.if 合并丢属性"，改了下标方式/挪了 copy 位置，
            # 报错文本一字不差——说明这两个猜测都不对，跟怎么下标无关。
            # 真正原因是 sK_buf/sV_buf 分配时没要求 16 字节对齐；分配处
            # 补上 alignment=16 后，这里恢复成最简单的动态下标就够了。
            sK_next = sK_buf[nxt, None, None]
            sV_next = sV_buf[nxt, None, None]
            cute.copy(tiled_copy_g2s_kv,
                      thr_copy_kv.partition_S(gK_next), thr_copy_kv.partition_D(sK_next))
            cute.copy(tiled_copy_g2s_kv,
                      thr_copy_kv.partition_S(gV_next), thr_copy_kv.partition_D(sV_next))
            cute.arch.cp_async_commit_group()
            # 此刻有 2 组在途：tile n_blk 自己那组（上一轮或 prologue 提交）
            # + 刚提交的 tile n_blk+1 那组。只等"只剩 1 组在途"，也就是
            # 等 tile n_blk 那组（更早提交的）落地即可，不用等新提交的。
            cute.arch.cp_async_wait_group(1)
        else:
            # 最后一块，没有下一块可发，把剩下那组（tile n_blk 自己）排空。
            cute.arch.cp_async_wait_group(0)

        cute.arch.sync_threads()   # cp.async 落地对其它线程可见

        sK = sK_buf[cur, None, None]
        sV = sV_buf[cur, None, None]

        # 7.2 GEMM1: S = Q @ K^T * scale（与 T7 相同，只是 sK 换成了
        #     双缓冲视图 sK_buf[cur]）
        tCrK = thr_mma_1.make_fragment_B(thr_mma_1.partition_B(sK))
        cpy_s2r_k = cute.make_tiled_copy_B(copy_atom_s2r, tiled_mma_1)
        thr_s2r_k = cpy_s2r_k.get_slice(tidx)
        cute.copy(cpy_s2r_k, thr_s2r_k.partition_S(sK), thr_s2r_k.retile(tCrK))

        tCrS.fill(0.0)
        cute.gemm(tiled_mma_1, tCrS, tCrQ, tCrK, tCrS)
        for i in range(cute.size(tCrS)):
            tCrS[i] = tCrS[i] * scale

        # 7.3 Online softmax 状态机（与 T7 完全相同，rowA/rowB 分组不变）
        row_max_a = cutlass.Float32(-NEG_INF)
        row_max_b = cutlass.Float32(-NEG_INF)
        for j in range(cute.size(tCrS) // 4):
            row_max_a = max(row_max_a,
                            cutlass.Float32(tCrS[4 * j + 0]),
                            cutlass.Float32(tCrS[4 * j + 1]))
            row_max_b = max(row_max_b,
                            cutlass.Float32(tCrS[4 * j + 2]),
                            cutlass.Float32(tCrS[4 * j + 3]))
        m_new_a = max(m_prev_a, warp_reduce_row_max(row_max_a))
        m_new_b = max(m_prev_b, warp_reduce_row_max(row_max_b))
        alpha_a = cute.arch.exp(m_prev_a - m_new_a)
        alpha_b = cute.arch.exp(m_prev_b - m_new_b)

        for j in range(cute.size(tCrO) // 4):
            tCrO[4 * j + 0] = tCrO[4 * j + 0] * alpha_a
            tCrO[4 * j + 1] = tCrO[4 * j + 1] * alpha_a
            tCrO[4 * j + 2] = tCrO[4 * j + 2] * alpha_b
            tCrO[4 * j + 3] = tCrO[4 * j + 3] * alpha_b

        row_sum_a = cutlass.Float32(0.0)
        row_sum_b = cutlass.Float32(0.0)
        for j in range(cute.size(tCrS) // 4):
            pa0 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 0]) - m_new_a)
            pa1 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 1]) - m_new_a)
            pb0 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 2]) - m_new_b)
            pb1 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 3]) - m_new_b)
            tCrS[4 * j + 0] = pa0
            tCrS[4 * j + 1] = pa1
            tCrS[4 * j + 2] = pb0
            tCrS[4 * j + 3] = pb1
            row_sum_a = row_sum_a + pa0 + pa1
            row_sum_b = row_sum_b + pb0 + pb1
        l_prev_a = alpha_a * l_prev_a + warp_reduce_row_sum(row_sum_a)
        l_prev_b = alpha_b * l_prev_b + warp_reduce_row_sum(row_sum_b)
        m_prev_a = m_new_a
        m_prev_b = m_new_b

        # 7.4 P 降级到 fp16 并写入 smem（sP 单缓冲，与 T7 相同）
        tCsP_w = thr_mma_1.partition_C(sP)
        for i in range(cute.size(tCrS)):
            tCsP_w[i] = cutlass.Float16(tCrS[i])
        cute.arch.sync_threads()

        # 7.5 GEMM2: O += P @ V（sV 换成双缓冲视图 sV_buf[cur]）
        tCrP = thr_mma_2.make_fragment_A(thr_mma_2.partition_A(sP))
        cpy_s2r_p = cute.make_tiled_copy_A(copy_atom_s2r, tiled_mma_2)
        thr_s2r_p = cpy_s2r_p.get_slice(tidx)
        cute.copy(cpy_s2r_p, thr_s2r_p.partition_S(sP), thr_s2r_p.retile(tCrP))

        sVT = cute.make_tensor(sV.iterator, cute.make_layout((D, BN), stride=(1, D)))
        tCrV = thr_mma_2.make_fragment_B(thr_mma_2.partition_B(sVT))
        cpy_s2r_v = cute.make_tiled_copy_B(copy_atom_s2r, tiled_mma_2)
        thr_s2r_v = cpy_s2r_v.get_slice(tidx)
        cute.copy(cpy_s2r_v, thr_s2r_v.partition_S(sVT), thr_s2r_v.retile(tCrV))

        cute.gemm(tiled_mma_2, tCrO, tCrP, tCrV, tCrO)

        # 【流水线优化 #4】这个 barrier 现在身兼两职：
        #   (a) 保护 sP 的 WAR（下一轮 7.4 会覆写 sP，必须等本轮 7.5 的
        #       S2R 读完）——这是 T7 里就有的原因；
        #   (b) 保护 sK/sV 双缓冲的 WAR——tile n_blk+2 会把 cp.async 写到
        #       stage (n_blk)%2，也就是本轮 cur 用过的那块，必须等本轮
        #       7.2/7.5 对 sK/sV[cur] 的 S2R 读全部完成。
        cute.arch.sync_threads()

    # ---------- 8. 收尾：归一化 + 写回（与 T7 相同） ----------
    inv_l_a = cutlass.Float32(1.0) / l_prev_a
    inv_l_b = cutlass.Float32(1.0) / l_prev_b
    for j in range(cute.size(tCrO) // 4):
        tCgO = thr_mma_2.partition_C(gO)
        tCgO[4 * j + 0] = cutlass.Float16(tCrO[4 * j + 0] * inv_l_a)
        tCgO[4 * j + 1] = cutlass.Float16(tCrO[4 * j + 1] * inv_l_a)
        tCgO[4 * j + 2] = cutlass.Float16(tCrO[4 * j + 2] * inv_l_b)
        tCgO[4 * j + 3] = cutlass.Float16(tCrO[4 * j + 3] * inv_l_b)


# ============================================================================
# Host 侧 JIT 入口 / Python 封装 / 测试 —— 和 T7 完全一致，原样带过来
# ============================================================================
@cute.jit
def flash_attn_fwd(
    mQ: cute.Tensor,
    mK: cute.Tensor,
    mV: cute.Tensor,
    mO: cute.Tensor,
    scale: cutlass.Float32,
):
    BH, S, d = mQ.shape
    grid = ((S + BM - 1) // BM, BH, 1)
    block = (NUM_THREADS, 1, 1)
    flash_attn_fwd_kernel(mQ, mK, mV, mO, scale).launch(grid=grid, block=block)


_compiled_cache = {}

def flash_attention_forward(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    B, H, S, d = Q.shape
    Skv = K.shape[2]
    assert d == D, f"当前 kernel 硬编码 head_dim={D}，输入是 {d}"
    assert S % BM == 0 and Skv % BN == 0, \
        "当前版本假设 S % BM == 0 且 S_kv % BN == 0（边界 predicate 未实现）"

    O = torch.empty_like(Q)

    Qf = Q.reshape(B * H, S, d).contiguous()
    Kf = K.reshape(B * H, Skv, d).contiguous()
    Vf = V.reshape(B * H, Skv, d).contiguous()
    Of = O.reshape(B * H, S, d)

    mQ = from_dlpack(Qf, assumed_align=16)
    mK = from_dlpack(Kf, assumed_align=16)
    mV = from_dlpack(Vf, assumed_align=16)
    mO = from_dlpack(Of, assumed_align=16)

    scale = cutlass.Float32(1.0 / (d ** 0.5))

    key = (B * H, S, Skv, d)
    if key not in _compiled_cache:
        _compiled_cache[key] = cute.compile(flash_attn_fwd, mQ, mK, mV, mO, scale)
    _compiled_cache[key](mQ, mK, mV, mO, scale)

    return Of.reshape(B, H, S, d)


def ref_attention(Q, K, V):
    return torch.nn.functional.scaled_dot_product_attention(Q, K, V)


def make_inputs(B, H, S, Skv, d, seed=0):
    gen = torch.Generator(device="cuda").manual_seed(seed)
    Q = (torch.randn(B, H, S, d, device="cuda", generator=gen) * 0.5).half()
    K = (torch.randn(B, H, Skv, d, device="cuda", generator=gen) * 0.5).half()
    V = (torch.randn(B, H, Skv, d, device="cuda", generator=gen) * 0.5).half()
    return Q, K, V


ATOL = 2e-2

TEST_SHAPES = [
    (1, 1, 64, 64, 64),
    (1, 2, 128, 128, 64),
    (2, 4, 256, 256, 64),
    (2, 8, 512, 512, 64),
    (1, 2, 128, 256, 64),
]


def test_correctness():
    print(f"{'shape':<28} {'max_abs_err':<14} {'result'}")
    print("-" * 48)
    for i, (B, H, S, Skv, d) in enumerate(TEST_SHAPES):
        Q, K, V = make_inputs(B, H, S, Skv, d, seed=i)
        O_ref = ref_attention(Q, K, V)
        O = flash_attention_forward(Q, K, V)
        assert O.shape == O_ref.shape and O.dtype == O_ref.dtype
        assert not torch.isnan(O).any(), f"[{i}] NaN in output"
        err = (O.float() - O_ref.float()).abs().max().item()
        ok = err < ATOL
        print(f"({B},{H},{S},{Skv},{d}){'':<16} {err:<14.6f} {'✅ PASS' if ok else '❌ FAIL'}")
        assert ok, f"shape {i} failed: err={err}"
    print("-" * 48)
    print("All correctness tests passed ✅")


def test_perf():
    import time
    B, H, S, d = 2, 16, 2048, 64
    Q, K, V = make_inputs(B, H, S, S, d, seed=42)
    flash_attention_forward(Q, K, V)
    torch.cuda.synchronize()
    N = 20
    t0 = time.perf_counter()
    for _ in range(N):
        flash_attention_forward(Q, K, V)
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / N
    flops = 4 * B * H * S * S * d
    print(f"[perf] {B}x{H}x{S}x{d}: {dt*1e3:.3f} ms, {flops/dt/1e12:.2f} TFLOPS")


test_correctness()
# test_perf()

shape                        max_abs_err    result
------------------------------------------------
(1,1,64,64,64)                 0.000000       ✅ PASS
(1,2,128,128,64)                 0.000122       ✅ PASS
(2,4,256,256,64)                 0.000061       ✅ PASS
(2,8,512,512,64)                 0.000061       ✅ PASS
(1,2,128,256,64)                 0.000061       ✅ PASS
------------------------------------------------
All correctness tests passed ✅


#### 优化方向：比 TMA 更该先做的(性价比更高,而且不依赖不成熟的 API):

- S2R 从逐元素换成 ldmatrix。 现在 K/P/V 的 S2R 拷贝走的是 16-bit CopyUniversalOp,一个线程一个元素地读——这是当前版本里最大的"技术债",换成 cute.nvgpu.warp.LdMatrix8x8x16bOp(warp 协作、一条指令读一整块)对 mma.sync 的吞吐影响比双缓冲流水线本身还直接。V 那边现在靠裸指针转置视图 + 逐元素读来绕过物理转置,换成 LdMatrix...(transpose=True) 应该能同时解决转置和向量化两个问题。
- 给 sK/sV/sP 加 swizzle。 现在都是纯行主序,mma.sync 访问 smem 时大概率有 bank conflict,allocate_tensor 的 swizzle 参数就是干这个的,加 swizzle 是同一个 allocator 上的自然延伸。
- cp.async 从 2 stage 加到 3~4 stage。 现在的双缓冲只能重叠"当前块计算"和"下一块搬运",更深的流水线能把 G2S 延迟藏得更彻底,尤其是 Skv 比较大的时候。加 stage 数本身工作量不大,踩的坑基本是这次已经趟过的(对齐、staged if/else)。
- 边界 predicate + causal mask。 现在两个都没做,如果要往 MHA/GQA 之类的真实推理场景靠拢,这个迟早要补。

In [5]:
# ============================================================================
# T7-async-v2 · Flash Attention Forward（Online Softmax + cp.async 4-stage
#             流水线 + ldmatrix S2R + smem swizzle，目标 sm_120 / 5070 Ti）
#   O = softmax(Q @ K^T * scale) @ V
#
# 相对 T7-async（2-stage 双缓冲版）的改动：
#
#   【优化 #6】S2R 全部从逐元素 16-bit CopyUniversalOp 换成 ldmatrix：
#       - A 操作数（Q、P）：LdMatrix8x8x16bOp x4 —— mma m16n8k16 的 A fragment
#         是 16x16 fp16 = 4 个 8x8 矩阵，一条 ldmatrix.x4 全读出来；
#       - B 操作数（K、V）：x2 —— B fragment 是 (N=8, K=16) = 2 个 8x8 矩阵；
#       - V 特殊：gemm2 的 B 是 (N=D, K=BN)，而 sV 物理上是 (BN, D) d 连续，
#         即 N 连续存储 —— 不是 mma 的原生朝向，必须用 transpose=True
#         （ldmatrix.trans，lane 地址指向 smem 里 16B 连续的行，寄存器映射
#         做转置）。这一步同时干掉了原来"裸指针转置视图 + 逐元素读"的绕法
#         的性能部分，但逻辑视图 sVT 还是要保留（见【坑 #E】）。
#       num_bits_per_copy 跟着 fragment 变：x4 -> 128bit/线程，x2 -> 64bit/线程。
#   【优化 #7】sK/sV/sP 加 128B swizzle（Swizzle<3,4,3>，fp16 行 64 元素=128B
#       的标准 swizzle）。这三块 smem 是 mma/ldmatrix 的热路径读方，
#       纯行主序下 mma.sync 访问模式撞 bank 的概率极高。sQ 只被读一次
#       （加载进 tCrQ 后全程驻留寄存器），swizzle 无收益，不加。
#   【优化 #8】cp.async 流水线 2 stage -> 4 stage：
#       - prologue 从"预取 1 块"推广为"预取 NSTAGES-1 块"（不足则能发几块发几块）；
#       - 主循环里改为"发 tile n_blk+NSTAGES-1，然后 cp_async_wait_group(NSTAGES-1)"
#         ——发完后在途共 NSTAGES 组，只等到剩 NSTAGES-1 组，即最老的
#         tile n_blk 那组落地，新发的 NSTAGES-1 组继续在途跟计算重叠；
#       - 尾段（没有下一块可发）只能 cp_async_wait_group(0) 排空。注意
#         cp.async.wait_group 的 N 是 PTX 立即数，不能传 n_tiles - n_blk - 1
#         这种运行时值；要想尾段也保留重叠，得按 constexpr 手工展开
#         NSTAGES-1 个分支，收益递减，这里不做。
#       - WAR 分析同 #4，只是推广：iter j 的 prefetch 写 stage (j+NSTAGES-1)%NSTAGES
#         = (j-1)%NSTAGES，即 iter j-1 刚用过的那块；iter j-1 结尾那个
#         sync_threads() 保证所有线程对它的 S2R 读已完成。NSTAGES 取 2 的
#         幂时 n_blk % NSTAGES 可被编译器降成位与，无除法开销。
#
# 【新踩的坑】
#   - 【坑 #E，数据正确性级别的坑】sV 加了 swizzle 之后，原来的转置视图
#       sVT = make_tensor(sV.iterator, make_layout((D, BN), stride=(1, D)))
#       是【错的】：make_tensor 用裸 stride 重建布局，swizzle 属性被绕过，
#       地址不再过 XOR —— 症状是结果整个错掉但不报任何错（比 IR
#       verification failed 难查多了）。修法：转置视图必须把同一个 swizzle
#       compose 回去（composition(sw, (D,BN):(1,D)) 在 (n,k) 处的 offset 是
#       sw(k*BN... = sw(k*64+n)，与 sV 在 (k,n) 处完全一致，数学上可验）。
#       —— 教训：swizzle 是布局属性，make_tensor 裸重建 = 静默丢属性，
#       和 #D 的 align 属于同一族问题，只不过这次不报错。
#   - ldmatrix 的 num_matrix_per_instr 要按 fragment 大小选：A 是 16x16
#       取 4，B 是 8x16 取 2。一开始 A/B 都写 4，make_tiled_copy_B 的
#       value layout 对不上 mma fragment，verification 报 shape mismatch。
#   - ldmatrix 要求每个 lane 给出的行地址 16B 对齐：sK/sV/sP 分配处
#       byte_alignment=16（#D 的教训直接沿用，这次一步到位）。
#
# 【还没实测过，仍是潜在风险点】
#   - smem 预算：NSTAGES=4 后 sK+sV = 2 * 4*64*64*2B = 64KB，加 sQ 8KB、
#     sP 8KB、sS 16KB，共 96KB。sm_120 单 block 动态 smem 上限约 99KB
#     （需要 opt-in），96KB 是贴着上限过的。如果 launch 报 smem 不足，
#     先把 NSTAGES 降到 3（总 80KB），或把 sS 改成 fp16 占位省 8KB。
#   - swizzle 布局过 make_tiled_copy_A/B 的 partition_S 兼容性检查依赖
#     DSL 的 layout 代数；万一 verification 报错，先把 swizzle 降成
#     64B 版（Swizzle<2,3,3>... 即 make_swizzle 对应组合）排查。
# ============================================================================

import torch
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

BM = 64          # Q 行 block
BN = 64          # K/V 列 block（内层循环步长）
D  = 64          # head_dim
NSTAGES = 4      # 【优化 #8】cp.async 流水线深度（2 的幂，% 降级为位与）

NUM_THREADS = 128   # = 4 warp，跟 tiled_mma 的 atom layout (4,1,1) 对应

NEG_INF = 1e30


@cute.jit
def warp_reduce_row_max(val: cutlass.Float32) -> cutlass.Float32:
    for offset in (1, 2):
        other = cute.arch.shuffle_sync_bfly(val, offset=offset)
        val = max(val, other)
    return val


@cute.jit
def warp_reduce_row_sum(val: cutlass.Float32) -> cutlass.Float32:
    for offset in (1, 2):
        other = cute.arch.shuffle_sync_bfly(val, offset=offset)
        val = val + other
    return val


@cute.kernel
def flash_attn_fwd_kernel(
    mQ: cute.Tensor,            # (BH, S,    d)
    mK: cute.Tensor,            # (BH, S_kv, d)
    mV: cute.Tensor,            # (BH, S_kv, d)
    mO: cute.Tensor,            # (BH, S,    d)
    scale: cutlass.Float32,
):
    # ---------- 0. 索引与块坐标 ----------
    bidx, bidy, _ = cute.arch.block_idx()
    tidx, _, _ = cute.arch.thread_idx()

    gQ_bh = mQ[bidy, None, None]
    gK_bh = mK[bidy, None, None]
    gV_bh = mV[bidy, None, None]
    gO_bh = mO[bidy, None, None]

    gQ = cute.local_tile(gQ_bh, (BM, D), (bidx, 0))
    gO = cute.local_tile(gO_bh, (BM, D), (bidx, 0))

    # ---------- 1. Shared Memory 布局 ----------
    # 【优化 #7】sK/sV/sP 加 Swizzle<3,4,3>（fp16、行 64 元素 = 128B 的
    # 标准 128B swizzle）。swizzle 走 allocate_tensor 的 swizzle 参数，
    # 不在 make_layout 上手工 compose —— 分配器和布局属性一处说清。
    sw128 = cute.make_swizzle(3, 4, 3)

    smem = cutlass.utils.SmemAllocator()
    sQ = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((BM, D), stride=(D, 1)),
        byte_alignment=16,   # ldmatrix 要求 128-bit 对齐；之前逐元素读不需要，
                             # 换 ldsm 后必须补上（坑 #D 的第三次现身）
    )

    sK_buf = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((NSTAGES, BN, D), stride=(BN * D, D, 1)),
        byte_alignment=16,   # 【坑 #D 沿用】ldmatrix / cp.async 都要 16B 对齐
        swizzle=sw128,       # 【优化 #7】
    )
    sV_buf = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((NSTAGES, BN, D), stride=(BN * D, D, 1)),
        byte_alignment=16,
        swizzle=sw128,
    )
    sP = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((BM, BN), stride=(BN, 1)),
        byte_alignment=16,
        swizzle=sw128,
    )
    sS = smem.allocate_tensor(cutlass.Float32, cute.make_layout((BM, BN), stride=(BN, 1)))

    # ---------- 2. TiledMMA（不变） ----------
    mma_atom = cute.make_mma_atom(
        cute.nvgpu.warp.MmaF16BF16Op(
            cutlass.Float16,
            acc_dtype=cutlass.Float32,
            shape_mnk=(16, 8, 16),
        )
    )
    tiled_mma_1 = cute.make_tiled_mma(mma_atom, (4, 1, 1))   # S = Q@K^T
    tiled_mma_2 = cute.make_tiled_mma(mma_atom, (4, 1, 1))   # O += P@V

    thr_mma_1 = tiled_mma_1.get_slice(tidx)
    thr_mma_2 = tiled_mma_2.get_slice(tidx)

    # ---------- 3. 拷贝原子 ----------
    # G2S 两个原子不变（Q 同步单次，K/V 走 cp.async 流水线）。
    copy_atom_g2s_sync = cute.make_copy_atom(
        op=cute.nvgpu.CopyUniversalOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )
    copy_atom_g2s_async = cute.make_copy_atom(
        op=cute.nvgpu.cpasync.CopyG2SOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )

    # 【优化 #6】S2R 原子：ldmatrix。
    #   注意：LdMatrix8x8x16bOp 构造签名是 (num_matrix_per_instr, transpose=False)，
    #   不收 dtype —— dtype 由外层 make_copy_atom 的 copy_internal_type 携带。
    #   - A 操作数 fragment 16x16 fp16 = 4 个 8x8 矩阵 -> x4 / 128bit；
    #   - B 操作数 fragment (8,16) fp16 = 2 个 8x8 矩阵 -> x2 / 64bit；
    #   - V 的 B 是 N 连续存储（非 mma 原生朝向）-> transpose=True。
    # 【坑 #F】LdMatrix8x8x16bOp 构造签名是 (transpose=False, num_matrices=4)：
    #   - 不收 dtype，dtype 由外层 make_copy_atom 的 copy_internal_type 携带；
    #   - 关键字名是 num_matrices（不是 num_matrix_per_instr）；
    #   - transpose 是第一个位置参数（布尔，不是 dtype）。
    copy_atom_ldsm_a = cute.make_copy_atom(
        op=cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=4),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )
    copy_atom_ldsm_b = cute.make_copy_atom(
        op=cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=2),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=64,
    )
    copy_atom_ldsm_b_trans = cute.make_copy_atom(
        op=cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=True, num_matrices=2),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=64,
    )



    tiled_copy_g2s_q = cute.make_tiled_copy_tv(
        copy_atom_g2s_sync, cute.make_layout((32, 4)), cute.make_layout((1, 8)))
    tiled_copy_g2s_kv = cute.make_tiled_copy_tv(
        copy_atom_g2s_async, cute.make_layout((32, 4)), cute.make_layout((1, 8)))

    # ---------- 4. Q 加载：同步 G2S -> ldmatrix S2R，全程常驻 ----------
    # sQ 不加 swizzle（只读一次，见文件头）；ldmatrix 对非 swizzle 布局照常工作。
    thr_copy_q = tiled_copy_g2s_q.get_slice(tidx)
    cute.copy(tiled_copy_g2s_q, thr_copy_q.partition_S(gQ), thr_copy_q.partition_D(sQ))
    cute.arch.sync_threads()

    tCrQ = thr_mma_1.make_fragment_A(thr_mma_1.partition_A(sQ))
    tiled_copy_s2r_q = cute.make_tiled_copy_A(copy_atom_ldsm_a, tiled_mma_1)
    thr_copy_s2r_q = tiled_copy_s2r_q.get_slice(tidx)
    cute.copy(tiled_copy_s2r_q,
              thr_copy_s2r_q.partition_S(sQ),
              thr_copy_s2r_q.retile(tCrQ))
    cute.arch.sync_threads()

    # ---------- 5. S fragment / O accumulator（不变） ----------
    tCrS = thr_mma_1.make_fragment_C(thr_mma_1.partition_C(sS))
    tCrO = thr_mma_2.make_fragment_C(thr_mma_2.partition_C(gO))
    tCrO.fill(0.0)

    # ---------- 6. Online Softmax 状态（不变） ----------
    m_prev_a = cutlass.Float32(-NEG_INF)
    m_prev_b = cutlass.Float32(-NEG_INF)
    l_prev_a = cutlass.Float32(0.0)
    l_prev_b = cutlass.Float32(0.0)

    n_tiles = (gK_bh.shape[0] + BN - 1) // BN
    thr_copy_kv = tiled_copy_g2s_kv.get_slice(tidx)

    # ---------- 6.5 Prologue：为 tile 0..NSTAGES-2 发起 cp.async 预取 ----------
    # 【优化 #8】预取 NSTAGES-1 块把流水线灌满；n_tiles 不足时能发几块发几块
    # （i 是编译期常量，i < n_tiles 是运行时分支，展开后就是最多 NSTAGES-1
    # 个 scf.if，没有额外成本）。
    for i in range(NSTAGES - 1):
        if i < n_tiles:
            gK_i = cute.local_tile(gK_bh, (BN, D), (i, 0))
            gV_i = cute.local_tile(gV_bh, (BN, D), (i, 0))
            cute.copy(tiled_copy_g2s_kv,
                      thr_copy_kv.partition_S(gK_i),
                      thr_copy_kv.partition_D(sK_buf[i, None, None]))
            cute.copy(tiled_copy_g2s_kv,
                      thr_copy_kv.partition_S(gV_i),
                      thr_copy_kv.partition_D(sV_buf[i, None, None]))
            cute.arch.cp_async_commit_group()

    # ---------- 7. 主循环：沿 N 维扫描 K/V（4-stage 软件流水线） ----------
    for n_blk in cutlass.range(n_tiles):
        cur = n_blk % NSTAGES
        prefetch = n_blk + NSTAGES - 1

        # 7.1 【优化 #8】先发 tile n_blk+NSTAGES-1 的 cp.async，跟本轮计算重叠。
        if prefetch < n_tiles:
            nxt = prefetch % NSTAGES
            gK_next = cute.local_tile(gK_bh, (BN, D), (prefetch, 0))
            gV_next = cute.local_tile(gV_bh, (BN, D), (prefetch, 0))
            sK_next = sK_buf[nxt, None, None]
            sV_next = sV_buf[nxt, None, None]
            cute.copy(tiled_copy_g2s_kv,
                      thr_copy_kv.partition_S(gK_next), thr_copy_kv.partition_D(sK_next))
            cute.copy(tiled_copy_g2s_kv,
                      thr_copy_kv.partition_S(gV_next), thr_copy_kv.partition_D(sV_next))
            cute.arch.cp_async_commit_group()
            # 发完后在途共 NSTAGES 组：tile n_blk（最老）+ 刚发的 NSTAGES-1 组。
            # 只等"剩 NSTAGES-1 组"，即 tile n_blk 落地，新发的继续在途。
            cute.arch.cp_async_wait_group(NSTAGES - 1)
        else:
            # 尾段：只剩 tile n_blk..n_tiles-1 共 n_tiles-n_blk <= NSTAGES-1 组，
            # 全部是本轮或后续轮要用的，直接排空。wait_group 的参数必须是
            # 编译期立即数（PTX 立即操作数），没法传 n_tiles - n_blk - 1
            # 这种运行时值精确只等一组 —— 想保住尾段重叠得 constexpr 展开
            # NSTAGES-1 个分支，收益递减，不做。
            cute.arch.cp_async_wait_group(0)

        cute.arch.sync_threads()   # cp.async 落地对其它线程可见

        sK = sK_buf[cur, None, None]
        sV = sV_buf[cur, None, None]

        # 7.2 GEMM1: S = Q @ K^T * scale
        # sK (BN, D) d 连续 = B 操作数 (N=BN, K=D) 的原生朝向（col B，K 连续），
        # ldmatrix 非 trans 直接喂。
        tCrK = thr_mma_1.make_fragment_B(thr_mma_1.partition_B(sK))
        cpy_s2r_k = cute.make_tiled_copy_B(copy_atom_ldsm_b, tiled_mma_1)
        thr_s2r_k = cpy_s2r_k.get_slice(tidx)
        cute.copy(cpy_s2r_k, thr_s2r_k.partition_S(sK), thr_s2r_k.retile(tCrK))

        tCrS.fill(0.0)
        cute.gemm(tiled_mma_1, tCrS, tCrQ, tCrK, tCrS)
        for i in range(cute.size(tCrS)):
            tCrS[i] = tCrS[i] * scale

        # 7.3 Online softmax 状态机（与 T7 完全相同，rowA/rowB 分组不变）
        row_max_a = cutlass.Float32(-NEG_INF)
        row_max_b = cutlass.Float32(-NEG_INF)
        for j in range(cute.size(tCrS) // 4):
            row_max_a = max(row_max_a,
                            cutlass.Float32(tCrS[4 * j + 0]),
                            cutlass.Float32(tCrS[4 * j + 1]))
            row_max_b = max(row_max_b,
                            cutlass.Float32(tCrS[4 * j + 2]),
                            cutlass.Float32(tCrS[4 * j + 3]))
        m_new_a = max(m_prev_a, warp_reduce_row_max(row_max_a))
        m_new_b = max(m_prev_b, warp_reduce_row_max(row_max_b))
        alpha_a = cute.arch.exp(m_prev_a - m_new_a)
        alpha_b = cute.arch.exp(m_prev_b - m_new_b)

        for j in range(cute.size(tCrO) // 4):
            tCrO[4 * j + 0] = tCrO[4 * j + 0] * alpha_a
            tCrO[4 * j + 1] = tCrO[4 * j + 1] * alpha_a
            tCrO[4 * j + 2] = tCrO[4 * j + 2] * alpha_b
            tCrO[4 * j + 3] = tCrO[4 * j + 3] * alpha_b

        row_sum_a = cutlass.Float32(0.0)
        row_sum_b = cutlass.Float32(0.0)
        for j in range(cute.size(tCrS) // 4):
            pa0 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 0]) - m_new_a)
            pa1 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 1]) - m_new_a)
            pb0 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 2]) - m_new_b)
            pb1 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 3]) - m_new_b)
            tCrS[4 * j + 0] = pa0
            tCrS[4 * j + 1] = pa1
            tCrS[4 * j + 2] = pb0
            tCrS[4 * j + 3] = pb1
            row_sum_a = row_sum_a + pa0 + pa1
            row_sum_b = row_sum_b + pb0 + pb1
        l_prev_a = alpha_a * l_prev_a + warp_reduce_row_sum(row_sum_a)
        l_prev_b = alpha_b * l_prev_b + warp_reduce_row_sum(row_sum_b)
        m_prev_a = m_new_a
        m_prev_b = m_new_b

        # 7.4 P 降级到 fp16 并写入 swizzle 后的 sP
        # （逐元素写经过 swizzled layout，地址自动过 XOR，正确性由布局代数保证；
        #   这里写入模式不是 ldmatrix 友好的主要原因 —— 读方 7.5 才是。）
        tCsP_w = thr_mma_1.partition_C(sP)
        for i in range(cute.size(tCrS)):
            tCsP_w[i] = cutlass.Float16(tCrS[i])
        cute.arch.sync_threads()

        # 7.5 GEMM2: O += P @ V
        # P：A 操作数 (M=BM, K=BN)，K 连续，原生朝向，ldmatrix x4。
        tCrP = thr_mma_2.make_fragment_A(thr_mma_2.partition_A(sP))
        cpy_s2r_p = cute.make_tiled_copy_A(copy_atom_ldsm_a, tiled_mma_2)
        thr_s2r_p = cpy_s2r_p.get_slice(tidx)
        cute.copy(cpy_s2r_p, thr_s2r_p.partition_S(sP), thr_s2r_p.retile(tCrP))

        # V：gemm2 的 B 是 (N=D, K=BN)，sV 物理上是 (BN, D) d 连续 = N 连续，
        # 非原生朝向 -> ldmatrix.trans。
        # 【坑 #E】逻辑转置视图必须把 swizzle compose 回去！裸
        # make_tensor(iterator, (D,BN):(1,D)) 会绕过 swizzle，地址错且不报错。
        # 验证：composition(sw, (D,BN):(1,D)) 在 (n,k) 处 offset = sw(n*1 + k*D)
        # = sw(k*64 + n)，与 sV 在 (k,n) 处逐元素一致。

        # V：gemm2 的 B 是 (N=D, K=BN)，sV 物理上是 (BN, D) d 连续 = N 连续，
        # 非原生朝向 -> ldmatrix.trans。
        # 【坑 #E 修正，2024-实测版】本 DSL 的 swizzle 挂在 pointer（memref
        # 类型）上，不在 layout 上：allocate_tensor(swizzle=...) 出来的
        # iterator 会对 layout 算出的线性 offset 自动套 XOR。因此：
        #   - 转置视图用裸 stride layout 即可，地址 = sw(n + k*64)，与 sV
        #     在 (k,n) 处逐元素一致 —— 我此前担心的"裸视图绕过 swizzle"不
        #     成立，真正的坑反而是反方向：往 layout 里再塞一份 swizzle
        #     （无论 composition 还是 make_composed_layout）都会触发
        #     MLIRError: cannot have both a swizzle on the pointer and a
        #     non-identity swizzle in the layout。
        # 规则：凡是从 swizzled 分配里切出来的 iterator，重建 tensor 时
        # layout 必须保持 identity swizzle，swizzle 由 pointer 负责。
        sVT = cute.make_tensor(
            sV.iterator,
            cute.make_layout((D, BN), stride=(1, D)),
        )


        tCrV = thr_mma_2.make_fragment_B(thr_mma_2.partition_B(sVT))
        cpy_s2r_v = cute.make_tiled_copy_B(copy_atom_ldsm_b_trans, tiled_mma_2)
        thr_s2r_v = cpy_s2r_v.get_slice(tidx)
        cute.copy(cpy_s2r_v, thr_s2r_v.partition_S(sVT), thr_s2r_v.retile(tCrV))

        cute.gemm(tiled_mma_2, tCrO, tCrP, tCrV, tCrO)

        # 【优化 #4 推广版】这个 barrier 现在身兼三职：
        #   (a) sP 的 WAR（下一轮 7.4 覆写 sP 前等本轮 7.5 读完）—— T7 就有；
        #   (b) sK/sV 的 WAR：iter j 的 prefetch 写 stage (j+NSTAGES-1)%NSTAGES
        #       = (j-1)%NSTAGES，即 iter j-1 刚用过的那块；本轮结尾这个
        #       barrier 保证上一轮所有线程的 ldmatrix 读已完成。NSTAGES
        #       越深，被保护的越"远"，但最近的那个 hazard 始终是 (j-1) 这一层，
        #       所以一个 barrier 就够，不需要随 NSTAGES 加同步点。
        #   (c) 顺带保证下一轮 7.1 的 cp.async 写与本轮全部 smem 读不踩踏。
        cute.arch.sync_threads()

    # ---------- 8. 收尾：归一化 + 写回（与 T7 相同，partition 提到循环外） ----------
    inv_l_a = cutlass.Float32(1.0) / l_prev_a
    inv_l_b = cutlass.Float32(1.0) / l_prev_b
    tCgO = thr_mma_2.partition_C(gO)
    for j in range(cute.size(tCrO) // 4):
        tCgO[4 * j + 0] = cutlass.Float16(tCrO[4 * j + 0] * inv_l_a)
        tCgO[4 * j + 1] = cutlass.Float16(tCrO[4 * j + 1] * inv_l_a)
        tCgO[4 * j + 2] = cutlass.Float16(tCrO[4 * j + 2] * inv_l_b)
        tCgO[4 * j + 3] = cutlass.Float16(tCrO[4 * j + 3] * inv_l_b)


# ============================================================================
# Host 侧 JIT 入口 / Python 封装 / 测试 —— 与 T7 完全一致
# ============================================================================
@cute.jit
def flash_attn_fwd(
    mQ: cute.Tensor,
    mK: cute.Tensor,
    mV: cute.Tensor,
    mO: cute.Tensor,
    scale: cutlass.Float32,
):
    BH, S, d = mQ.shape
    grid = ((S + BM - 1) // BM, BH, 1)
    block = (NUM_THREADS, 1, 1)
    flash_attn_fwd_kernel(mQ, mK, mV, mO, scale).launch(grid=grid, block=block)


_compiled_cache = {}

def flash_attention_forward(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    B, H, S, d = Q.shape
    Skv = K.shape[2]
    assert d == D, f"当前 kernel 硬编码 head_dim={D}，输入是 {d}"
    assert S % BM == 0 and Skv % BN == 0, \
        "当前版本假设 S % BM == 0 且 S_kv % BN == 0（边界 predicate 未实现）"

    O = torch.empty_like(Q)

    Qf = Q.reshape(B * H, S, d).contiguous()
    Kf = K.reshape(B * H, Skv, d).contiguous()
    Vf = V.reshape(B * H, Skv, d).contiguous()
    Of = O.reshape(B * H, S, d)

    mQ = from_dlpack(Qf, assumed_align=16)
    mK = from_dlpack(Kf, assumed_align=16)
    mV = from_dlpack(Vf, assumed_align=16)
    mO = from_dlpack(Of, assumed_align=16)

    scale = cutlass.Float32(1.0 / (d ** 0.5))

    key = (B * H, S, Skv, d)
    if key not in _compiled_cache:
        _compiled_cache[key] = cute.compile(flash_attn_fwd, mQ, mK, mV, mO, scale)
    _compiled_cache[key](mQ, mK, mV, mO, scale)

    return Of.reshape(B, H, S, d)


def ref_attention(Q, K, V):
    return torch.nn.functional.scaled_dot_product_attention(Q, K, V)


def make_inputs(B, H, S, Skv, d, seed=0):
    gen = torch.Generator(device="cuda").manual_seed(seed)
    Q = (torch.randn(B, H, S, d, device="cuda", generator=gen) * 0.5).half()
    K = (torch.randn(B, H, Skv, d, device="cuda", generator=gen) * 0.5).half()
    V = (torch.randn(B, H, Skv, d, device="cuda", generator=gen) * 0.5).half()
    return Q, K, V


ATOL = 2e-2

TEST_SHAPES = [
    (1, 1, 64, 64, 64),
    (1, 2, 128, 128, 64),
    (2, 4, 256, 256, 64),
    (2, 8, 512, 512, 64),
    (1, 2, 128, 256, 64),
]


def test_correctness():
    print(f"{'shape':<28} {'max_abs_err':<14} {'result'}")
    print("-" * 48)
    for i, (B, H, S, Skv, d) in enumerate(TEST_SHAPES):
        Q, K, V = make_inputs(B, H, S, Skv, d, seed=i)
        O_ref = ref_attention(Q, K, V)
        O = flash_attention_forward(Q, K, V)
        assert O.shape == O_ref.shape and O.dtype == O_ref.dtype
        assert not torch.isnan(O).any(), f"[{i}] NaN in output"
        err = (O.float() - O_ref.float()).abs().max().item()
        ok = err < ATOL
        print(f"({B},{H},{S},{Skv},{d}){'':<16} {err:<14.6f} {'✅ PASS' if ok else '❌ FAIL'}")
        assert ok, f"shape {i} failed: err={err}"
    print("-" * 48)
    print("All correctness tests passed ✅")


def test_perf():
    import time
    B, H, S, d = 2, 16, 2048, 64
    Q, K, V = make_inputs(B, H, S, S, d, seed=42)
    flash_attention_forward(Q, K, V)
    torch.cuda.synchronize()
    N = 20
    t0 = time.perf_counter()
    for _ in range(N):
        flash_attention_forward(Q, K, V)
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / N
    flops = 4 * B * H * S * S * d
    print(f"[perf] {B}x{H}x{S}x{d}: {dt*1e3:.3f} ms, {flops/dt/1e12:.2f} TFLOPS")


test_correctness()
# test_perf()


shape                        max_abs_err    result
------------------------------------------------
(1,1,64,64,64)                 0.000000       ✅ PASS
(1,2,128,128,64)                 0.000122       ✅ PASS
(2,4,256,256,64)                 0.000061       ✅ PASS
(2,8,512,512,64)                 0.000061       ✅ PASS
(1,2,128,256,64)                 0.000061       ✅ PASS
------------------------------------------------
All correctness tests passed ✅


In [8]:
# ============================================================================
# T7-flash-v3 · Flash Attention Forward
#   Online Softmax + cp.async 4-stage 流水线 + ldmatrix S2R + smem swizzle
#   + 边界 predicate + causal mask（目标 sm_120 / 5070 Ti）
#   O = softmax(Q @ K^T * scale) @ V
#
# 相对 v2（ldmatrix + swizzle + 4-stage 版）的改动：
#
#   【边界 #9】去掉 host 侧 "S % BM == 0 and Skv % BN == 0" 硬约束：
#     - Q 侧：最后一个 Q block 的同步 G2S 改为逐元素 guarded 拷贝，
#       越界行写 0（Q 行垃圾虽不会写回，但清零可以彻底绕开 NaN 推理）；
#     - O 写回：按行坐标 predicate，越界行不写。
#   【causal #10】运行时 flag（Int32，0/1），只做自回归语义（要求 S == Skv）：
#     - 免费收益：block bidx 的 Q 行最大列号是 bidx*BM+BM-1，之后的 K tile
#       整块跳过 —— n_tiles_eff = min(n_tiles_full, ceil((bidx+1)*BM/BN))，
#       grid 后半的 block 工作量近似减半；
#     - 对角线块内部：S 算完、scale 之后、softmax 状态机之前，按全局
#       (row, col) 坐标把 col > row 的位置写成 -NEG_INF。mask 的是逻辑值，
#       与 smem 里搬了什么无关，softmax 状态机零改动。
#   【残差 tile 的流水线处理】kv 残差 tile（Skv % BN != 0 时的最后一块）
#     不能对 gmem 越界行发起任何访存，cp.async 的逐行 predicate 又没有
#     验证过的 API，所以残差 tile 走同步逐元素拷贝：
#     - 不 commit_group —— 每线程组计数不受影响，wait_group 语义不乱；
#     - 会在循环里引入"这个 tile 没有在途组"的 hole，处理规则：
#         * 消费残差 tile 的迭代：同步拷贝 + barrier，无需 wait（此时在途
#           组必为 0，见循环内推导）；
#         * prefetch 目标是残差 tile 的迭代：跳过发起，wait_group(NSTAGES-2)
#           只等当前 tile（立即数约束下能做到的最好重叠）；
#     - 实测前提：wait_group 的 N 必须是编译期立即数，NSTAGES-2 / NSTAGES-1 /
#       0 都是 Python 常量，满足。
#   【V 残差清零，K 顺手】残差同步拷贝对越界行写 0 而不是留垃圾：
#     K 行垃圾会被 S-mask 覆盖成 -inf，无害；但 V 行垃圾若为 NaN/Inf 位型，
#     gemm2 里 P=0 × NaN = NaN 会直接污染 tCrO，mask 救不回来 —— 必须清零。
#
# 【已知风险点（写代码时已尽量压到 primitives-only，但未实测）】
#   - 残差拷贝里用了 Int32 的 // % （idx // D、idx % D）和 smem/gmem 的
#     动态二维下标 sK_cur[r, c] —— 若这版 DSL 不支持会报 TypeError/
#     verification error，备选是把扁平 idx 循环改成 (行, 列) 两层循环。
#   - n_tiles_eff 用了 Python 内建 min() 作用于 DSL Int32 —— 与 T7 血统里
#     max() 用法同族，理论上没问题；万一不支持，换成算术 select。
#   - causal 走运行时分支，mask 每元素两次比较 + 偶发跳转；想消掉就按
#     causal 拆两个编译变体（工厂闭包捕获 Python bool），暂不做。
# ============================================================================

import torch
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

BM = 64          # Q 行 block
BN = 64          # K/V 列 block（内层循环步长）
D  = 64          # head_dim
NSTAGES = 4      # cp.async 流水线深度（2 的幂，% 降级为位与）

NUM_THREADS = 128   # = 4 warp，跟 tiled_mma 的 atom layout (4,1,1) 对应

NEG_INF = 1e30


@cute.jit
def warp_reduce_row_max(val: cutlass.Float32) -> cutlass.Float32:
    for offset in (1, 2):
        other = cute.arch.shuffle_sync_bfly(val, offset=offset)
        val = max(val, other)
    return val


@cute.jit
def warp_reduce_row_sum(val: cutlass.Float32) -> cutlass.Float32:
    for offset in (1, 2):
        other = cute.arch.shuffle_sync_bfly(val, offset=offset)
        val = val + other
    return val


@cute.kernel
def flash_attn_fwd_kernel(
    mQ: cute.Tensor,            # (BH, S,    d)
    mK: cute.Tensor,            # (BH, S_kv, d)
    mV: cute.Tensor,            # (BH, S_kv, d)
    mO: cute.Tensor,            # (BH, S,    d)
    scale: cutlass.Float32,
    causal: cutlass.Int32,      # 1 = 自回归 causal mask（要求 S == S_kv）
):
    # ---------- 0. 索引与块坐标 ----------
    bidx, bidy, _ = cute.arch.block_idx()
    tidx, _, _ = cute.arch.thread_idx()

    gQ_bh = mQ[bidy, None, None]
    gK_bh = mK[bidy, None, None]
    gV_bh = mV[bidy, None, None]
    gO_bh = mO[bidy, None, None]

    gQ = cute.local_tile(gQ_bh, (BM, D), (bidx, 0))
    gO = cute.local_tile(gO_bh, (BM, D), (bidx, 0))

    # 静态 shape 信息（from_dlpack 进来的是编译期常量，可直接参与
    # trace 期 Python 分支，用来在可整除的场景下彻底不生成 predicate 代码）
    S_full   = gQ_bh.shape[0]
    Skv_full = gK_bh.shape[0]
    q_residue  = (S_full % BM != 0)     # Python bool，trace 期
    kv_has_res = (Skv_full % BN != 0)   # Python bool，trace 期
    n_q_blocks = (S_full + BM - 1) // BM
    n_tiles_full = (Skv_full + BN - 1) // BN
    # 残差 tile 下标；kv 无残差时置 -1（tile 下标不可能为 -1，比较恒假，
    # 动态分支会被运行时折叠掉，且分支体永远不执行，正确性无碍）
    n_tiles_last = n_tiles_full - 1 if kv_has_res else -1

    # ---------- 1. Shared Memory 布局 ----------
    # sK/sV/sP 带 Swizzle<3,4,3>（坑 #E 最终版：swizzle 挂 pointer 上，
    # 转置视图必须保持裸 identity layout）；sQ 只读一次不 swizzle。
    # 全部 byte_alignment=16（坑 #D：ldmatrix/cp.async 都要 128-bit 对齐）。
    sw128 = cute.make_swizzle(3, 4, 3)

    smem = cutlass.utils.SmemAllocator()
    sQ = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((BM, D), stride=(D, 1)),
        byte_alignment=16,
    )
    sK_buf = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((NSTAGES, BN, D), stride=(BN * D, D, 1)),
        byte_alignment=16,
        swizzle=sw128,
    )
    sV_buf = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((NSTAGES, BN, D), stride=(BN * D, D, 1)),
        byte_alignment=16,
        swizzle=sw128,
    )
    sP = smem.allocate_tensor(
        cutlass.Float16,
        cute.make_layout((BM, BN), stride=(BN, 1)),
        byte_alignment=16,
        swizzle=sw128,
    )
    sS = smem.allocate_tensor(cutlass.Float32, cute.make_layout((BM, BN), stride=(BN, 1)))

    # ---------- 2. TiledMMA ----------
    mma_atom = cute.make_mma_atom(
        cute.nvgpu.warp.MmaF16BF16Op(
            cutlass.Float16,
            acc_dtype=cutlass.Float32,
            shape_mnk=(16, 8, 16),
        )
    )
    tiled_mma_1 = cute.make_tiled_mma(mma_atom, (4, 1, 1))   # S = Q@K^T
    tiled_mma_2 = cute.make_tiled_mma(mma_atom, (4, 1, 1))   # O += P@V

    thr_mma_1 = tiled_mma_1.get_slice(tidx)
    thr_mma_2 = tiled_mma_2.get_slice(tidx)

    # ---------- 3. 拷贝原子 ----------
    # 【坑 #F】LdMatrix8x8x16bOp(transpose=False, num_matrices=4)：不收 dtype，
    # dtype 由 make_copy_atom 的 copy_internal_type 携带；第一个位置参数是
    # transpose（布尔）。A fragment 16x16 = 4 个 8x8 -> x4/128bit；
    # B fragment (8,16) = 2 个 8x8 -> x2/64bit；V 非原生朝向 -> transpose=True。
    copy_atom_g2s_sync = cute.make_copy_atom(
        op=cute.nvgpu.CopyUniversalOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )
    copy_atom_g2s_async = cute.make_copy_atom(
        op=cute.nvgpu.cpasync.CopyG2SOp(),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )
    copy_atom_ldsm_a = cute.make_copy_atom(
        op=cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=4),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=128,
    )
    copy_atom_ldsm_b = cute.make_copy_atom(
        op=cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=False, num_matrices=2),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=64,
    )
    copy_atom_ldsm_b_trans = cute.make_copy_atom(
        op=cute.nvgpu.warp.LdMatrix8x8x16bOp(transpose=True, num_matrices=2),
        copy_internal_type=cutlass.Float16,
        num_bits_per_copy=64,
    )

    tiled_copy_g2s_q = cute.make_tiled_copy_tv(
        copy_atom_g2s_sync, cute.make_layout((32, 4)), cute.make_layout((1, 8)))
    tiled_copy_g2s_kv = cute.make_tiled_copy_tv(
        copy_atom_g2s_async, cute.make_layout((32, 4)), cute.make_layout((1, 8)))

    # ---------- 4. Q 加载：【边界 #9】最后一个 Q block 走 guarded 同步拷贝 ----------
    thr_copy_q = tiled_copy_g2s_q.get_slice(tidx)
    if q_residue and bidx == n_q_blocks - 1:   # Python and：q_residue=False 时不生成分支
        # 逐元素 guarded：越界行写 0，规避垃圾值进 softmax
        for k in range(BM * D // NUM_THREADS):
            idx = tidx + k * NUM_THREADS
            r = idx // D
            c = idx % D
            g_row = bidx * BM + r
            if g_row < S_full:
                sQ[r, c] = gQ_bh[g_row, c]
            else:
                sQ[r, c] = cutlass.Float16(0.0)
    else:
        cute.copy(tiled_copy_g2s_q, thr_copy_q.partition_S(gQ), thr_copy_q.partition_D(sQ))
    cute.arch.sync_threads()

    tCrQ = thr_mma_1.make_fragment_A(thr_mma_1.partition_A(sQ))
    tiled_copy_s2r_q = cute.make_tiled_copy_A(copy_atom_ldsm_a, tiled_mma_1)
    thr_copy_s2r_q = tiled_copy_s2r_q.get_slice(tidx)
    cute.copy(tiled_copy_s2r_q,
              thr_copy_s2r_q.partition_S(sQ),
              thr_copy_s2r_q.retile(tCrQ))
    cute.arch.sync_threads()

    # ---------- 5. S fragment / O accumulator ----------
    tCrS = thr_mma_1.make_fragment_C(thr_mma_1.partition_C(sS))
    tCrO = thr_mma_2.make_fragment_C(thr_mma_2.partition_C(gO))
    tCrO.fill(0.0)

    # identity 坐标张量：mask / store predicate 共用（gemm 示例的标准做法）
    cS = cute.make_identity_tensor((BM, BN))
    tCcS = thr_mma_1.partition_C(cS)
    cO = cute.make_identity_tensor((BM, D))
    tCcO = thr_mma_2.partition_C(cO)

    # ---------- 6. Online Softmax 状态 ----------
    m_prev_a = cutlass.Float32(-NEG_INF)
    m_prev_b = cutlass.Float32(-NEG_INF)
    l_prev_a = cutlass.Float32(0.0)
    l_prev_b = cutlass.Float32(0.0)

        # 【causal #10】免费收益：本 block 只需要 K 列 <= bidx*BM+BM-1
    # 注意：n_tiles_eff 必须在 staged if 之前定义初值 —— staged 分支是独立
    # region，region 内首次定义的变量不会泄漏到外面（实测 NameError，DSL 的
    # suggestion 也是这么说的：变量在 for/if/while 之前要有初值，分支内
    # 赋值才能通过 yield 带出来）。同时用 Int32 统一类型，避免 yield 时
    # 两分支类型不一致（Python int vs DSL Int32）。
    causal_tiles = (bidx * BM + BM + BN - 1) // BN   # 运行时 Int32
    n_tiles_eff = cutlass.Int32(n_tiles_full)        # 初值 = 非 causal 的上界
    if causal != 0:
        n_tiles_eff = min(causal_tiles, n_tiles_eff)


    need_mask = kv_has_res or (causal != 0)   # trace 期：可整除且非 causal 时整段不生成
    thr_copy_kv = tiled_copy_g2s_kv.get_slice(tidx)

    # ---------- 6.5 Prologue：预取 tile 0..NSTAGES-2 ----------
    for i in range(NSTAGES - 1):
        skip = (kv_has_res and i == n_tiles_last)   # Python bool，残差 tile 不发 cp.async
        if not skip:
            if i < n_tiles_eff:
                gK_i = cute.local_tile(gK_bh, (BN, D), (i, 0))
                gV_i = cute.local_tile(gV_bh, (BN, D), (i, 0))
                cute.copy(tiled_copy_g2s_kv,
                          thr_copy_kv.partition_S(gK_i),
                          thr_copy_kv.partition_D(sK_buf[i, None, None]))
                cute.copy(tiled_copy_g2s_kv,
                          thr_copy_kv.partition_S(gV_i),
                          thr_copy_kv.partition_D(sV_buf[i, None, None]))
                cute.arch.cp_async_commit_group()

    # ---------- 7. 主循环（4-stage 流水线 + 残差/越界处理） ----------
    for n_blk in cutlass.range(n_tiles_eff):
        cur = n_blk % NSTAGES

        if kv_has_res and n_blk == n_tiles_last:
            # ==== 残差 tile：同步逐元素 guarded 拷贝（不能对越界行访存）====
            # 此时在途组必为 0：上一迭代 pf = n_blk+NSTAGES-1 > 最后可预取
            # tile，走了 wait_group(0) 排空 —— 无需再 wait。
            # V 越界行必须清零（P=0 × NaN = NaN 会污染累加器，见文件头）。
            row0 = n_blk * BN
            sK_cur = sK_buf[cur, None, None]
            sV_cur = sV_buf[cur, None, None]
            for k in range(BN * D // NUM_THREADS):
                idx = tidx + k * NUM_THREADS
                r = idx // D
                c = idx % D
                g_row = row0 + r
                if g_row < Skv_full:
                    sK_cur[r, c] = gK_bh[g_row, c]
                    sV_cur[r, c] = gV_bh[g_row, c]
                else:
                    sK_cur[r, c] = cutlass.Float16(0.0)
                    sV_cur[r, c] = cutlass.Float16(0.0)
        else:
            # ==== 常规流水线：发 tile n_blk+NSTAGES-1，等 tile n_blk 落地 ====
            pf = n_blk + NSTAGES - 1
            if pf == n_tiles_last:
                # prefetch 目标是残差 tile：跳过发起。此时在途 = tile
                # n_blk..n_tiles_last-1 共 NSTAGES-1 组，wait(NSTAGES-2)
                # 恰好等最老的 tile n_blk 落地（立即数约束下的最好重叠）。
                cute.arch.cp_async_wait_group(NSTAGES - 2)
            else:
                if pf < n_tiles_eff:
                    gK_next = cute.local_tile(gK_bh, (BN, D), (pf, 0))
                    gV_next = cute.local_tile(gV_bh, (BN, D), (pf, 0))
                    cute.copy(tiled_copy_g2s_kv,
                              thr_copy_kv.partition_S(gK_next),
                              thr_copy_kv.partition_D(sK_buf[pf % NSTAGES, None, None]))
                    cute.copy(tiled_copy_g2s_kv,
                              thr_copy_kv.partition_S(gV_next),
                              thr_copy_kv.partition_D(sV_buf[pf % NSTAGES, None, None]))
                    cute.arch.cp_async_commit_group()
                    # 发完在途 NSTAGES 组，只等 tile n_blk 那组
                    cute.arch.cp_async_wait_group(NSTAGES - 1)
                else:
                    # 尾段没有下一块可发：排空（wait_group 参数必须是立即数，
                    # 尾段重叠需要编译期展开分支，收益递减不做）
                    cute.arch.cp_async_wait_group(0)

        cute.arch.sync_threads()   # cp.async / 同步拷贝对其它线程可见

        sK = sK_buf[cur, None, None]
        sV = sV_buf[cur, None, None]

        # 7.2 GEMM1: S = Q @ K^T * scale
        tCrK = thr_mma_1.make_fragment_B(thr_mma_1.partition_B(sK))
        cpy_s2r_k = cute.make_tiled_copy_B(copy_atom_ldsm_b, tiled_mma_1)
        thr_s2r_k = cpy_s2r_k.get_slice(tidx)
        cute.copy(cpy_s2r_k, thr_s2r_k.partition_S(sK), thr_s2r_k.retile(tCrK))

        tCrS.fill(0.0)
        cute.gemm(tiled_mma_1, tCrS, tCrQ, tCrK, tCrS)
        for i in range(cute.size(tCrS)):
            tCrS[i] = tCrS[i] * scale

        # 7.25 【#9/#10】统一 mask 插点：scale 之后、row_max 之前。
        # mask 的是逻辑值，与 smem 搬了什么无关；越界列的 smem 垃圾值
        # 算出的 S 也在这里被覆盖，所以 predicate 和 mask 缺一不可。
        if need_mask:
            for i in range(cute.size(tCrS)):
                row = bidx * BM + tCcS[i][0]
                col = n_blk * BN + tCcS[i][1]
                if col >= Skv_full:
                    tCrS[i] = cutlass.Float32(-NEG_INF)
                else:
                    if causal != 0:
                        if col > row:
                            tCrS[i] = cutlass.Float32(-NEG_INF)

        # 7.3 Online softmax 状态机（与 T7 完全相同）
        row_max_a = cutlass.Float32(-NEG_INF)
        row_max_b = cutlass.Float32(-NEG_INF)
        for j in range(cute.size(tCrS) // 4):
            row_max_a = max(row_max_a,
                            cutlass.Float32(tCrS[4 * j + 0]),
                            cutlass.Float32(tCrS[4 * j + 1]))
            row_max_b = max(row_max_b,
                            cutlass.Float32(tCrS[4 * j + 2]),
                            cutlass.Float32(tCrS[4 * j + 3]))
        m_new_a = max(m_prev_a, warp_reduce_row_max(row_max_a))
        m_new_b = max(m_prev_b, warp_reduce_row_max(row_max_b))
        alpha_a = cute.arch.exp(m_prev_a - m_new_a)
        alpha_b = cute.arch.exp(m_prev_b - m_new_b)

        for j in range(cute.size(tCrO) // 4):
            tCrO[4 * j + 0] = tCrO[4 * j + 0] * alpha_a
            tCrO[4 * j + 1] = tCrO[4 * j + 1] * alpha_a
            tCrO[4 * j + 2] = tCrO[4 * j + 2] * alpha_b
            tCrO[4 * j + 3] = tCrO[4 * j + 3] * alpha_b

        row_sum_a = cutlass.Float32(0.0)
        row_sum_b = cutlass.Float32(0.0)
        for j in range(cute.size(tCrS) // 4):
            pa0 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 0]) - m_new_a)
            pa1 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 1]) - m_new_a)
            pb0 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 2]) - m_new_b)
            pb1 = cute.arch.exp(cutlass.Float32(tCrS[4 * j + 3]) - m_new_b)
            tCrS[4 * j + 0] = pa0
            tCrS[4 * j + 1] = pa1
            tCrS[4 * j + 2] = pb0
            tCrS[4 * j + 3] = pb1
            row_sum_a = row_sum_a + pa0 + pa1
            row_sum_b = row_sum_b + pb0 + pb1
        l_prev_a = alpha_a * l_prev_a + warp_reduce_row_sum(row_sum_a)
        l_prev_b = alpha_b * l_prev_b + warp_reduce_row_sum(row_sum_b)
        m_prev_a = m_new_a
        m_prev_b = m_new_b

        # 7.4 P 降级到 fp16 写入 swizzled sP
        tCsP = thr_mma_1.partition_C(sP)
        for i in range(cute.size(tCrS)):
            tCsP[i] = cutlass.Float16(tCrS[i])
        cute.arch.sync_threads()

        # 7.5 GEMM2: O += P @ V
        tCrP = thr_mma_2.make_fragment_A(thr_mma_2.partition_A(sP))
        cpy_s2r_p = cute.make_tiled_copy_A(copy_atom_ldsm_a, tiled_mma_2)
        thr_s2r_p = cpy_s2r_p.get_slice(tidx)
        cute.copy(cpy_s2r_p, thr_s2r_p.partition_S(sP), thr_s2r_p.retile(tCrP))

        # V 转置视图：裸 identity layout（坑 #E 最终版：pointer 携带 swizzle，
        # 任何往 layout 里再组合 swizzle 的写法都会报 MLIRError）
        sVT = cute.make_tensor(
            sV.iterator,
            cute.make_layout((D, BN), stride=(1, D)),
        )
        tCrV = thr_mma_2.make_fragment_B(thr_mma_2.partition_B(sVT))
        cpy_s2r_v = cute.make_tiled_copy_B(copy_atom_ldsm_b_trans, tiled_mma_2)
        thr_s2r_v = cpy_s2r_v.get_slice(tidx)
        cute.copy(cpy_s2r_v, thr_s2r_v.partition_S(sVT), thr_s2r_v.retile(tCrV))

        cute.gemm(tiled_mma_2, tCrO, tCrP, tCrV, tCrO)

        # 循环尾 barrier：sP 的 WAR + sK/sV 双缓冲的 WAR（分析见 v2 头注释 #4）
        cute.arch.sync_threads()

    # ---------- 8. 收尾：归一化 + predicate 写回（【边界 #9】） ----------
    inv_l_a = cutlass.Float32(1.0) / l_prev_a
    inv_l_b = cutlass.Float32(1.0) / l_prev_b
    tCgO = thr_mma_2.partition_C(gO)
    if q_residue:
        for j in range(cute.size(tCrO) // 4):
            row_a = bidx * BM + tCcO[4 * j + 0][0]
            if row_a < S_full:
                tCgO[4 * j + 0] = cutlass.Float16(tCrO[4 * j + 0] * inv_l_a)
                tCgO[4 * j + 1] = cutlass.Float16(tCrO[4 * j + 1] * inv_l_a)
            row_b = bidx * BM + tCcO[4 * j + 2][0]
            if row_b < S_full:
                tCgO[4 * j + 2] = cutlass.Float16(tCrO[4 * j + 2] * inv_l_b)
                tCgO[4 * j + 3] = cutlass.Float16(tCrO[4 * j + 3] * inv_l_b)
    else:
        for j in range(cute.size(tCrO) // 4):
            tCgO[4 * j + 0] = cutlass.Float16(tCrO[4 * j + 0] * inv_l_a)
            tCgO[4 * j + 1] = cutlass.Float16(tCrO[4 * j + 1] * inv_l_a)
            tCgO[4 * j + 2] = cutlass.Float16(tCrO[4 * j + 2] * inv_l_b)
            tCgO[4 * j + 3] = cutlass.Float16(tCrO[4 * j + 3] * inv_l_b)


# ============================================================================
# Host 侧 JIT 入口 / Python 封装 / 测试
# ============================================================================
@cute.jit
def flash_attn_fwd(
    mQ: cute.Tensor,
    mK: cute.Tensor,
    mV: cute.Tensor,
    mO: cute.Tensor,
    scale: cutlass.Float32,
    causal: cutlass.Int32,
):
    BH, S, d = mQ.shape
    grid = ((S + BM - 1) // BM, BH, 1)   # grid 本来就按 ceil 算，天然支持非整除
    block = (NUM_THREADS, 1, 1)
    flash_attn_fwd_kernel(mQ, mK, mV, mO, scale, causal).launch(grid=grid, block=block)


_compiled_cache = {}

def flash_attention_forward(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                            causal: bool = False) -> torch.Tensor:
    B, H, S, d = Q.shape
    Skv = K.shape[2]
    assert d == D, f"当前 kernel 硬编码 head_dim={D}，输入是 {d}"
    if causal:
        assert S == Skv, "causal 仅实现自回归语义（S == S_kv）；cross-attention 的 causal 语义未定义"

    O = torch.empty_like(Q)

    Qf = Q.reshape(B * H, S, d).contiguous()
    Kf = K.reshape(B * H, Skv, d).contiguous()
    Vf = V.reshape(B * H, Skv, d).contiguous()
    Of = O.reshape(B * H, S, d)

    mQ = from_dlpack(Qf, assumed_align=16)
    mK = from_dlpack(Kf, assumed_align=16)
    mV = from_dlpack(Vf, assumed_align=16)
    mO = from_dlpack(Of, assumed_align=16)

    scale = cutlass.Float32(1.0 / (d ** 0.5))
    causal_i = cutlass.Int32(1 if causal else 0)

    key = (B * H, S, Skv, d, causal)
    if key not in _compiled_cache:
        _compiled_cache[key] = cute.compile(flash_attn_fwd, mQ, mK, mV, mO, scale, causal_i)
    _compiled_cache[key](mQ, mK, mV, mO, scale, causal_i)

    return Of.reshape(B, H, S, d)


def ref_attention(Q, K, V, causal=False):
    return torch.nn.functional.scaled_dot_product_attention(Q, K, V, is_causal=causal)


def make_inputs(B, H, S, Skv, d, seed=0):
    gen = torch.Generator(device="cuda").manual_seed(seed)
    Q = (torch.randn(B, H, S, d, device="cuda", generator=gen) * 0.5).half()
    K = (torch.randn(B, H, Skv, d, device="cuda", generator=gen) * 0.5).half()
    V = (torch.randn(B, H, Skv, d, device="cuda", generator=gen) * 0.5).half()
    return Q, K, V


ATOL = 2e-2

TEST_SHAPES = [
    (1, 1, 64, 64, 64),      # 整除基线
    (1, 2, 128, 128, 64),
    (2, 4, 256, 256, 64),
    (2, 8, 512, 512, 64),
    (1, 2, 128, 256, 64),
    (1, 1, 96, 160, 64),     # 【#9】S、Skv 均不整除（残差 32 / 32）
    (2, 4, 200, 168, 64),    # 【#9】残差 8 / 40
    (1, 2, 300, 300, 64),    # 【#9+#10】causal 下 n_tiles_eff 截断 + 残差叠加
]


def test_correctness():
    print(f"{'shape':<28} {'causal':<8} {'max_abs_err':<14} {'result'}")
    print("-" * 60)
    for causal in (False, True):
        for i, (B, H, S, Skv, d) in enumerate(TEST_SHAPES):
            if causal and S != Skv:
                continue
            Q, K, V = make_inputs(B, H, S, Skv, d, seed=i)
            O_ref = ref_attention(Q, K, V, causal=causal)
            O = flash_attention_forward(Q, K, V, causal=causal)
            assert O.shape == O_ref.shape and O.dtype == O_ref.dtype
            assert not torch.isnan(O).any(), f"[{i}] NaN in output"
            err = (O.float() - O_ref.float()).abs().max().item()
            ok = err < ATOL
            print(f"({B},{H},{S},{Skv},{d}){'':<16} {str(causal):<8} "
                  f"{err:<14.6f} {'✅ PASS' if ok else '❌ FAIL'}")
            assert ok, f"shape {i} causal={causal} failed: err={err}"
    print("-" * 60)
    print("All correctness tests passed ✅")


def test_perf(causal=False):
    import time
    B, H, S, d = 2, 16, 2048, 64
    Q, K, V = make_inputs(B, H, S, S, d, seed=42)
    flash_attention_forward(Q, K, V, causal=causal)
    torch.cuda.synchronize()
    N = 20
    t0 = time.perf_counter()
    for _ in range(N):
        flash_attention_forward(Q, K, V, causal=causal)
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / N
    # causal 实际 flops 约减半，这里仍打满算力的名义值，只做前后对照
    flops = 4 * B * H * S * S * d
    print(f"[perf causal={causal}] {B}x{H}x{S}x{d}: {dt*1e3:.3f} ms, "
          f"{flops/dt/1e12:.2f} TFLOPS(nominal)")


test_correctness()
# test_perf(causal=False)
# test_perf(causal=True)



shape                        causal   max_abs_err    result
------------------------------------------------------------
(1,1,64,64,64)                 False    0.000000       ✅ PASS
(1,2,128,128,64)                 False    0.000122       ✅ PASS
(2,4,256,256,64)                 False    0.000061       ✅ PASS
(2,8,512,512,64)                 False    0.000061       ✅ PASS
(1,2,128,256,64)                 False    0.000061       ✅ PASS
(1,1,96,160,64)                 False    0.000061       ✅ PASS
(2,4,200,168,64)                 False    0.000122       ✅ PASS
(1,2,300,300,64)                 False    0.000061       ✅ PASS
(1,1,64,64,64)                 True     0.000000       ✅ PASS
(1,2,128,128,64)                 True     0.000122       ✅ PASS
(2,4,256,256,64)                 True     0.000122       ✅ PASS
(2,8,512,512,64)                 True     0.000244       ✅ PASS
(1,2,300,300,64)                 True     0.000122       ✅ PASS
----------------------------------------------------

### T8 · FA BWD 前置：Delta 预计算（热身）
- **描述**：`delta[i] = Σⱼ dO[i,j] · O[i,j]`，即 `rowsum(dO ∘ O)`，shape 为 `(M,)`。这是 BWD 中唯一需要预计算的量（可存 HBM，也可融合进 kernel），后续每行一个标量。
- **要点**：
  1. `dO`、`O` 同为 `(M, d)`，按 `T6 RMSNorm` 的方式各自 tile + partition，两 fragment 对齐遍历，逐元素乘后本线程累加；
  2. 一个 warp 内 redux add（或复用 T7 的 shuffle 逻辑，注意这里不是按 MMA 布局而是普通行布局）；
  3. 写回 `delta[m]`，每行一个标量。
- **验收**：与 `(dO * O).sum(-1)` 对照，atol=1e-3。
- **结构提示**：独立小 kernel，5 分钟能写完，但它决定了 T9 里 `dS` 的数值正确性——先把它做对。

### T9 · Flash Attention Backward（综合）
- **描述**：给定 `Q, K, V, O, dO, delta, scale`，计算 `dQ, dK, dV`。核心是**重算 S、P**（BWD 不存中间矩阵，靠重新做一遍 GEMM 1 换取显存 O(N)）：每行 block 上 `dV = P^T @ dO`，`dS = P ∘ (dP - delta)`，`dQ = dS @ K`，`dK = dS^T @ Q`。
- **要点**：
  1. 每个 CTA 取一个 K/V block（沿 N 切），循环扫 Q——注意 **BWD 的循环方向与 FWD 相反**（FWD 冻结 Q 扫 K，BWD 冻结 K/V 扫 Q/dO）；
  2. 重算前向：`S = Q @ K^T * scale`，`P = exp(S - delta_row 对应的 m)`。两种方案：a) 直接加载预存的 `m/l`；b) 简化实现存 `delta` 并用 `P = exp(S*scale - delta/2 + log l)` 的代数变形——**推荐先存 `m` 和 `l` 两个 `(M,)` 向量，最不易错**；
  3. `dP = dO @ V^T`（复用 `tiled_mma`，`dO` 当 A、`V` 当 B）；`dS = P * (dP - delta[m])`，`delta[m]` 需按行广播到 fragment——广播时同样要按 MMA 行布局取对应 lane 的值；
  4. 三个输出 GEMM 的角色映射：`dV = P^T @ dO`（P 当 A 但**转置 layout**，shape 换位 stride 沿用）、`dQ += dS @ K`（跨 block 累加，需原子加或分 CTA 写）、`dK = dS^T @ Q`；
  5. `P`、`dS` 喂 MMA 前记得 `.to(cutlass.Float16)`；`dQ` 累加用 FP32，最后统一乘 `scale`（或把 scale 提前乘进 `dS`）。
- **验收**：对 `torch.autograd.grad` 逐项对照 `dQ/dK/dV`，atol=2e-2；建议先做 `BM=BN=64`、无因果 mask 的最小版本。
- **结构提示**：T7 + T8 的叠加，难度在**一次循环里管理四个 partitioned tensor（Q/dO 常驻、K/V 流式、dS 瞬态）+ 两次转置 layout + 行广播**。建议拆步：先只出 `dV`（不需要 delta 广播），再补 `dK`，最后补 `dQ`——每一步都用 autograd 验一次，别等全写完再调。